# Crawl corpus ViBioMIR trên Kaggle

Notebook tự chứa code crawl, không cần clone repo hoặc upload file Python.
Phần chẩn đoán đã tích hợp: Output tự tạo `<run>_diagnostics.zip` để debug.
Bật **Internet** trong cài đặt notebook; dùng CPU, không cần GPU.
Chỉnh ô **1. Cấu hình**, rồi chạy các ô theo thứ tự.

ID bắt đầu và kết thúc đều được tính vào khoảng chọn. Đây là **ID gốc**,
không phải số thứ tự dòng. Giữ nguyên ID/URL nguồn, không dịch hoặc sinh lại text.
Trích HTML/PDF có text; Selenium thử lại URL lỗi và render JavaScript. Chưa có OCR.
Dùng chung `priority_probe.py` với notebook 97 hostname: header trình duyệt,
timeout/retry có ngân sách, giữ lỗi theo từng lần thử và kiểm tra challenge.
Tài liệu nguồn: [ViBioMIR](https://huggingface.co/datasets/AIGuruTinix/ViBioMIR).


## 1. Cấu hình — chỉnh ID tại đây

Mặc định chọn ID **583–1582**. Để thử trước, đặt `MAX_NEW = 10`;
sau đó đổi thành `None` và chạy lại ô crawl để làm tiếp.
Giữ nguyên cấu hình tải trang/browser và phiên bản code khi resume cùng run.
Với khoảng lớn, chia thành nhiều run nhỏ: crawler hiện tại nạp danh sách công việc vào RAM.

`SOURCE_PARQUET` để trống sẽ tải nguồn đã chốt từ Hugging Face.
Nếu đã thêm Parquet vào Kaggle Input, điền đường dẫn `/kaggle/input/.../links_corpus.parquet`.
`RESUME_RUN_PATH` tùy chọn trỏ tới **thư mục run đã giải nén** từ notebook này trong Kaggle Input.
Để nâng cấp run cũ hoặc thử lại lỗi, đặt `RECOVERY_RUN_PATH` và chọn `RUN_TAG` mới.
Recovery dùng lại raw 2xx đã kiểm tra hash, chạy lại extractor/QC, rồi crawl URL còn lỗi.
`RESPECT_ROBOTS = False` tải trực tiếp giống `domain_samples`; đặt `True` để
kiểm tra robots trước các request trang. Trạng thái robots không được chuyển sang Selenium.


In [1]:
# ===== CHỈNH CẤU HÌNH TẠI ĐÂY =====
ID_FROM = 583
ID_TO = 1582              # Bao gồm ID này
WORKERS = 4
DELAY = 1.0
ATTEMPTS = 3
MAX_NEW = None            # None: toàn bộ còn lại; 10: thêm tối đa 10 URL
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36'
PAGE_CONNECT_TIMEOUT = 20.0
PAGE_READ_TIMEOUT = 45.0
PAGE_REQUEST_BUDGET = 180.0
RESPECT_ROBOTS = False    # False: giống domain_samples; True: kiểm tra robots
ROBOTS_ATTEMPTS = 3
ROBOTS_CONNECT_TIMEOUT = 15.0
ROBOTS_READ_TIMEOUT = 30.0
ROBOTS_RETRY_BUDGET = 90.0
ROBOTS_FAILURE_TTL = 30.0
BROWSER_TEST_MODE = 'failed_only'  # 'off', 'failed_only', 'all'
BROWSER_USER_AGENT = None  # Dùng User-Agent thực của Chrome
BROWSER_LOAD_TIMEOUT = 45.0
BROWSER_WAIT_SECONDS = 15.0
BROWSER_ATTEMPTS = 2
CHROME_BINARY = None      # Selenium Manager tự tìm/cài Chrome; hoặc đường dẫn có sẵn

SOURCE_PARQUET = None     # None: tự tải; hoặc '/kaggle/input/.../links_corpus.parquet'
RESUME_RUN_PATH = None    # None: mới/tiếp tục session; hoặc thư mục run trong Kaggle Input
RECOVERY_RUN_PATH = None  # Run cũ để reuse raw thành công, thử lại URL lỗi trong run mới
RUN_TAG = None            # None: ids_<ID_FROM>_<ID_TO>_synced; đổi khi muốn run mới

from pathlib import Path
import json
import shutil
import sys

if sys.version_info < (3, 11):
    raise RuntimeError('Code crawl yêu cầu Python 3.11 trở lên.')
WORKDIR = Path('/kaggle/working')
DATASET_REVISION = '0148f6f80ffafed5c005af6d506ccfd9d3fb47a7'
SOURCE_SHA256 = 'dee1153f5b9d9010dea18e1b7724b4915ff847b711d036bec8b933486802a971'
SOURCE_URL = f'https://huggingface.co/datasets/AIGuruTinix/ViBioMIR/resolve/{DATASET_REVISION}/links_corpus.parquet'

if type(ID_FROM) is not int or type(ID_TO) is not int or ID_FROM < 0 or ID_TO < ID_FROM:
    raise ValueError('ID_FROM và ID_TO phải là số nguyên; ID_TO >= ID_FROM >= 0.')
if type(WORKERS) is not int or WORKERS < 1 or type(ATTEMPTS) is not int or ATTEMPTS < 1:
    raise ValueError('WORKERS và ATTEMPTS phải là số nguyên dương.')
if not isinstance(DELAY, (int, float)) or not 0 <= DELAY < float('inf'):
    raise ValueError('DELAY phải hữu hạn và không âm.')
if MAX_NEW is not None and (type(MAX_NEW) is not int or MAX_NEW < 1):
    raise ValueError('MAX_NEW phải là None hoặc số nguyên dương.')
if RECOVERY_RUN_PATH and RESUME_RUN_PATH:
    raise ValueError('Chỉ chọn RECOVERY_RUN_PATH hoặc RESUME_RUN_PATH.')
tag = RUN_TAG or f'ids_{ID_FROM}_{ID_TO}_synced' + ('_recovery' if RECOVERY_RUN_PATH else '')
if not isinstance(tag, str) or tag in ('.', '..') or any(separator in tag for separator in ('/', chr(92))):
    raise ValueError('RUN_TAG chỉ là tên thư mục, không phải đường dẫn.')
RUN_DIR = WORKDIR / 'runs' / tag
SOURCE_PATH = WORKDIR / 'data' / 'links_corpus.parquet'
RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
if RESUME_RUN_PATH:
    previous = Path(RESUME_RUN_PATH)
    if not (previous / 'manifest.json').is_file():
        raise ValueError('RESUME_RUN_PATH phải chứa manifest.json, records/ và raw/.')
    if not RUN_DIR.exists():
        shutil.copytree(previous, RUN_DIR)
    else:
        print('Đã có output trong session; dùng run hiện tại, không copy đè.')
if (RUN_DIR / 'manifest.json').exists():
    if (RUN_DIR / 'crawl_config.json').exists() and not (RUN_DIR / 'corpus_config.json').exists():
        raise ValueError('Run cũ cần nâng cấp: chọn RUN_TAG mới và dùng RECOVERY_RUN_PATH.')
    saved = json.loads((RUN_DIR / 'manifest.json').read_text(encoding='utf-8'))
    if saved.get('id_start_inclusive') != ID_FROM or saved.get('id_stop_exclusive') != ID_TO + 1:
        raise ValueError('Khoảng ID không khớp run đã có. Chọn RUN_TAG mới.')
print(f'ID: {ID_FROM} đến {ID_TO}, bao gồm cả hai đầu')
print(f'Output: {RUN_DIR}')


ID: 583 đến 1582, bao gồm cả hai đầu
Output: \kaggle\working\runs\ids_583_1582_synced


## 2. Cài dependencies

Lượt mới dùng phiên bản tương thích mà pip chọn trong môi trường Kaggle,
Phiên bản thực tế được lưu cùng run.
Khi khôi phục run, notebook cài lại dependencies trực tiếp đã ghi nhận.


In [2]:
import subprocess
import importlib.metadata

package_names = ['pyarrow', 'requests', 'trafilatura', 'pymupdf', 'tld']
if BROWSER_TEST_MODE != 'off':
    package_names.append('selenium')
requirements_path = RUN_DIR / 'notebook_requirements.txt'
if requirements_path.exists():
    install_args = ['-r', str(requirements_path)]
else:
    install_args = package_names
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'logs').mkdir(exist_ok=True)
installation = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', *install_args],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace',
)
with (RUN_DIR / 'logs' / 'pip_install.log').open('a', encoding='utf-8') as log:
    log.write(installation.stdout or '')
if installation.stdout:
    print(installation.stdout)
if installation.returncode:
    raise subprocess.CalledProcessError(installation.returncode, installation.args, output=installation.stdout)
versions = {name: importlib.metadata.version(name) for name in package_names}
print(json.dumps(versions, indent=2))



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip

{
  "pyarrow": "25.0.1",
  "requests": "2.34.2",
  "trafilatura": "2.3.0",
  "pymupdf": "1.28.2",
  "tld": "0.13.2",
  "selenium": "4.50.0"
}


## 3. Chuẩn bị code crawl

Ô dưới giải nén snapshot nhúng trong notebook vào output; không tải code từ mạng.
Các file `.py` sinh ra có thể mở để xem. Snapshot có sẵn của run được giữ nguyên khi resume.
Snapshot code được lưu trực tiếp trong notebook này.


In [3]:
import base64
import hashlib
import zlib

EMBEDDED_FILES = {'audit_snapshot.py': 'c-pO5TW=f36@EA1|1hCj?6M_SmpDdCkun_HL5#$S;W$MBh1iUChvY<i^X#r9Mk3H64}B`oAO(t`2oNVIf;1?aqJ5Gc`k)~HL;WTF&dls`MUk>y)PP0q&biIGd~@co)oQ&v8bvY@OeTdGD_Km}f)rnqf`yWglQb`6kTafy_|M8D;=3Y>M97rbm*Rls6CnyVN)_WQQ2Zcbs!Y6At9AXFjI&e~jH_|RRW7by8>uvAK??SukZF=LOK~?Xp;6Tw<Bw%lKTLQ&iRAA5xrigibN%UyJm8tA*FMVA#Hz`7L2Q+ZM|iBs6n2xw^T)Y>e45wa%R~lgC_-NF>({K*=n%>qhSbE_lq;1Uc-ZOF5)OfZ*;8V>ek~LusNkVXCbfIluQB{nq9|3O4fdjxhq+$$h|Vv>Vd07d%4IShIAt;F{=#Xu=hA{S&UjF`M7SW(VUG7vDigP1y_cja=8^nZxXwFo+;uukGkSRz$--?4&L`X@js>AM1Rj<*U%O=}yoopr<yhp9Fc%PWsIItm)D;`l8Q|Sf<1$8SJ<=;~i0~<yQ7#O8KTHGPZ#P*z9)>=*Jg(F2rey)^m3Uf8CBnfIRf?+_Op<nU?47nf&BOqC-eItnmr*g;SYNjSz?H*PD@T8jq};XZP>p3uBbQ!#c(yAsPvi)Gz_JcAQLZtkdhJn+9ZM^Xk1Lv-$5|vaMdY+i^mnI)fYvs9Wl)RWv6Q}vfHSln)-mET7wlI&D#d-IpmLq}G_8&ri)9}3BA8SL7&$O?r;&PjTNg0uF=1DkwmdOS9ubDX?4eO34l~p^P-4K2)Q%%Vn9T5{RFN~($`mOxN@muyv{V6^mOZuV_--WJh7UA>GwiUDjDQ#UV8<oV9p=ace<O)xtTeaV!?stXem2#w=l;La9<5<UuCL_0h^NC;<Sa=Grd?fa<dK1b%JWJDsR}8_b_^OAr)Ng5f~q<#EGb+&j?!J%`I*h3%Dbrjl1I<+traq>H*UEadt;!DMB-XfopSVC`|!%mKQ0sKAuNhSAxIV-8~}hE&{?Z^a4+9t2VB}SB8pfr5y4&tj4JZVHv!Lu_RO%E3O>~?y*{#=SlnB~Jd!|^<PPl(atjiJoHF+gUOa^laLD<2!HY7dHE%3{6$|g%ARR?~OcXoA*<2AEX*qD%`2^Zd(@pemHw~w-g-$Uq6yzW!^1O;JY^gbi3pIaHjs*)6r$b{omterkAdOm&`N2gb=+#*>Yh708P>nm<$NJ1C13<3d4n1A_!h(~1{gU=P0n5u6L}f)D@DLbP<OdP}yzBh*(Y-gimmIH=CkY6MO^D$Ah~uR2E6{S<l&2QCE;AQOP!`e2?$LRl7Ya-|0y>h|J0x|eLbHeiixxv>!!J!X3!SmlT4~9i1Zcr1xeqhQ*S68>icl7*>SpM*wCi1A%!l!LZcTRT^Llt$C2C2fFQ@V_5Gwm0)di72FF2_C5bJH7Bpu>yuQ0K}Q#UHY(^+H<bKCO`m|ZOsM%7-O2filrg9(mo;GaP-{ca_r;h?G%YF~t!%h(64W^?eBc_SiZ@H=TDmTak_p+Yg=C|fHTOOkVLg29%d$Foc%p=(V7yFV(UXxcEYy;QMDvK1;&C^FEeydm3JA6{jP^`ky+5eM|*wI8h{?~vfib-cATO|>+_mje^?@$&`Bc)U^^nu<N0*4$*+M2V5pwzm!VxsqnGoSPiKKzrrJc~S6SLcOAo<H2OlPN~J0B;Kee#1~#JxC$n&az60fEF2x9TRpsed>G}2x7+UR-cOEqZr>QTk^Ut$sNREBnjww+pfjY+`fTI1b)7$}J+~srSLBlB)>UO&<=jsL<(p>MS>fv&>_HL=l0bLA-6#$Au#u5aKs=xd-F|TTuzT-5?65@aqp=^0L?}*&yk0FP!WIvy_LghW#T_+cH_rrf891hI#WWK1MUE7Y^*3BZ_3~2Cufad^LX~DdL7%%~pUvK4#~wTsc;|z}B2wWI?q?Q?ip50JU9irynsA9K!j0ZbTHwAlIr~>KX2ID%SuW_74E?X4pFBfT`H;&kwkOX|KY^ze*RdF%pMDyU-&0n#6kfY=tdXVj!*CAI7rU88Za<U(5H}qa59mIgh~oZX7Ntt4<2X;dCMxu;Fkkzxe6WLjk)z|jezASr$NhNQUNCJPEI46#%BG`{giBK8$6W1|nU6f$TQqceqeiWW!-GI%$VBH#Z7_<oEeJBJ9khAPI9}j{6K#v0gY&}DB2QlJi5{x<mI|40%nmx-k<P3@9~HXwrN@zeoC~(Ca2dlTQ{CH|oS<E53^BJS>Qt*HyjrOmdMw#wi(wU786R~kM^Y2g?YVO)Lv@P@Kr;A2jT+Enk&8L=*^png$)UutZnoN9A{T(EujRQ<J)AkU!G-;~feFasv5Xos*D5-G7d-Z&8XDrBuZ#Bf@_L%bXMR!3l9RQ=*4u2Gt-pYtJV-MEfzF+>NMjAO2+kT1xH+2J(k`O=7ER95D8zRFgr2Z5o8z9dMEs=0)!H5kelvsC?RBTRj`9-ysc#d}UPZkFj0(*m<LXI5SjHJ935KYHHassCCMbCz<=_ntFzjHjbhHdM+qAqeL|CTLlv6c<Xee|nP&X8-Wh&e3=Ni`M7f0*sRr)I!nb*6kF5Nn>!lFSFi=byuOa~E<cS8VneJ!mA0^0MK8WJ>^nWXnvf4Da{(SPgmb}MlmT$mCwd$OD~)#{k8%F?teG%{1!0jmcJngP=;>&|*8K)Iud0gjlH<ZctM;qX|uU(df*7nlHv`EA-f68_Y8-QMMUb7$H42ryIdbOD+%Nu=U$Kpdt_xxemXva{J(q_#s(pcdDbYKU>+3V7rvWxR_q(glpL5{<`%n!kakba^|@teFyZ3>*EPap?agVRZ;rCoK7h0_5~x0V>gaQ(s=U`+7<qaWJ&@Wox@1ivsg`x+95V(9$ySR)=Z(4pkctDV3FZf6csG1PsTX@N^YE8sV<i+qjh-(!3l7c1yX1vLDN&w|O%=+@eTyCqj-VAoFg`LMWF}ezc475vsRAjA<l8b|VOatqen3?5wx3305nHz5|Cahn4Cml(>rWX%D|ngj@8d3!QjVh%SCiDB~pWZHyGd|E)34v<WkW^ypv$?{#xrH$@N6-2=t5trTb3C`u3XQ;#Rpg9%#PEwiPY%`MG5I@yk(B2~Q`zr6M8tZy3ACC-^_Y(JXF^OJv<%^nT)#)`8qB`y-@r@t#}c<HZ!zMpMBJU{uaWXafkOyt>LF$xrbpPQQ=yL*26c>ytJ&%l57rD^7h^V9FB1xM3&_Eo}mNjU1M!vam6P0mlgHsmIp9{b@BXJ1Vb`5(XM)$$(fEO0d70So6Re~F3x8xBi;{Qms(3(0=?^z2zOVPkpr%<HdZ+s^RP0Itsa23}yCZ>J~rhZ(Q>eWs$Ie61R!QLAqgW4mw9jqQ38#QyV}Ke8jRyB83}Ocy8+mH}QgR2GV8E!*y6;KB4bW8hE<InZE!b3$djx=jvD#^w3xXBd9WJ=w8tLR<Y=%<P1ujbObBHA)yo&E{QOHV4IDi+J9a8#9Rw##XwUM%>i_!tllDLFLaaSyXy?z;apk8um<otxH=Cgs9tWh$VDbJZ>Au1FbI{w)MS;M6HIRIMhINgS|B`fYkoBXp&Kzj^TZ)i>dJ1`;Xsddz16iKW9}eFbHfmud}tIMwdJDS{JI$=`tE^?pKxZ!>5$HUkA<ld+*IVU5q9dx*yVi-#UE^;=h9z>5m^HN($ZQ8PmXFaDyG$Oxt;0dpU%?SJF~e>w=~0osIg!rpUC{E0SH*CoY5F#NRicJbb(1r&WqB_M*et68RNQ=^m?gd0o!!*;2^bvb8~m)i7be13D<K7lL#5>{$S~Ir}zi#v?iVa>@>ADFNr-5|$~zal&c9K!F6qGYp4GbHZYcXgO+$Mdzn~qT8>%9V2PJ#V6XK35D+Do2iXp0v5@_5{q!&Kk%XZ;S4x{)XhETvZ2R>vtO`d_8uu}?);djk6E|dW%_RqZ-!N>S69gqD2~wN%|LUc8+?2-$2Z<~%+0Uy3y--TdHPg$>uzl>Z-)v*1IdS%df*TBE<e$pd~k1zeYjAz)jIwCeNY43etj4K<xdFKlL;@kK!$z(Hfdr#(WV#V8pXzO!v)m1^ZvaDPyEOCAHDndi5G{fP$LenXkRd1e}Vgh{+-wU%8Z`X_Y(|-d|!9lzE8h7`+jxv$MKD+Ob8o20+M7T#}MSM0xkV}_4RB21)<}oc>',
 'corpus_runner_snapshot.py': 'c-o~JU2ojR@%?_qRzX3YbD|`FH9dz9s;neN6w8(57{&F#P~_enBOb|cNu5>s{zLmv^tC}y1ZWYo=%fA62M75d`X`;)4|10uCp(b=-`(!)%<jzWeD2oP)=8O=VnGR8_&E(pm@fS&Az7ZDQ$mw@lu$<eBqUjR9!G3Jo;=>pa>{6a86|TPq<K~{5-ezNk)=^mFlTFPYY;87G%tvs&oe(~bTG@)CBVip4T>mD7?A)6X_*u>ACZvG{4y@WC@9q8fL*Gu;v<gEowA7HYBW{fzGi8nzAyb^A?RiB4b~`H&u~J1XID&pEfzWTLpaM|@a5sj>67E9#3hzE*LGee3C*`J_pHI=!%y~~ADx}T?Dc@apH%`AJbzA;!Wt9ni}X)X9Q!*DoLyqSh>|c}G4k|`?Cm<cj|dDry#I*2e0YCI_OmReFX;K_QL*#j-fx_H4~hNxXJ=m=jYu3_Q1U78G98kG1ssCz?A-$d_>Y|WGe3{y5^E$j5z^Sd0{VN-h**A6&}CLI_<aw47SWQXCGKu=cNgaGKX9fnd|rlgilciU!Kgq363sw19*8~V>p$^hM%5%A^wNyW4koYFP=lZx4N~A5Agk%`J&=St{?sZIgSO-z?zo;S@++^}_;2;OnIFYvPQ9Xt#j>^B_IVB>01++IkfMYTkcM8G#8=j+o;JkzG)*YX1)TTZd6fA1l^Qm&#;LJg;H`#p<wu3bC@>;q^7!?n#<aVm!Fn(lfc6u9eC97fH<%sd{wl_M0mz4API&1j(TuVJ^iR$@d<&@iHh_^GVksC02dyDr7tJd88Jyo+_-vuz1vw5KTSM|Ix1utUa~_z5yAK{(({Uxf+>aO~U;1%L5A!_DZR?bS62uIaz)$8hbR_jvR4hn0IVa6%V$&o@@%Fh^S<JRSf?fEGumZTTtVv3?z|kkTm>-<kG1#UdnWZ_wZv?ECScgZ(5eI%onLV5;KEy!pI-gikXifRLoG%i{*ZlklA_s<>j{LaH;wbP7Dwl&<Wa`YItYu!|)y12GM&zL(-4Sdg4)KS8msHVq!EpuS1O$HBx)xC43~SEHEp@ykE%-WzzKmGGZ2bUYOQ^<ji@1?aRYDAv=GX$zfoo?Y$>E6TBV`V9<b1RBeW|ldIKXrhBWKn@<vM_eJI_y!$TDJJ#DfLc8Qjd;WUHVrxhkDfi?Q5&N&Xsq&FMF#xJ3N|u_O?a)6A-nD848(5#v1VXSBTYo9QXT&`r({0?OD(I^z-Cu$0jRF!Dn>V%h^t0hlaX?rfwe4^4TagS7<)ea0!X5R}=L$XMYQC38*6u2?1{FNQ2NH|(<R6&ocw`CA|mV?v=M*rF)1PFQWi+DwU}Qh68PeN7DYxIvK*cR=1kMexi+<#cCTO>Nofx}%lmNU`V|qYFT(f@FjI*<_i5(m7^;-pVmeHzgO0^@<C`R84i{hQe+>vCt6WLR7^l*w?6BT7ActXR+wZBKL!WhM-~KL<?}vUU8LC*D7FYXHhKUN9bxGwk%xXIQmYh#AAwCjROA36jg%95)Bv;3-$63Hk-p1!JoHOTaNdyg{UoK03%}rH-|ttv`tMkL13r5L-HXp<<tQ)BKL-Z`O3Jj@~EIjpr9;GuTN2SzTfl;q|#vI6l3X4_E)*nEX~T;7fHZ|NQR|l9#9YPV9+AtnTZ7!sDSVik_Ki8w+;$WW}7foHwpoTu1j>>!QBc~c~g)X_^#`<;0S<&@k2NsmQ0Nl20{~72-abkAwz^Z)@Z8#ZSJVcEoe?(0E`V)!&?(2C9Nr?D@<K3=+#*F;qY6;Q1AtJXd75iC13~W#^FQchWO3!6F5-$1FIg>jgWZ{tZzy`j*2S{ff|{`{#;<>CP;k|wp;hj-H5xKiLtB)o1|tUGmWvHGc>Rpmed-qTLMZ~lP0z)&#L*1$;~OoQ`VRJ$KK=T#|IV%<7#ON7*_T;jV@)jMh$b%=O&WN)1j^=6AQ5V0N{b##chL4x%$=Rs-w(?REsGZ9~}>K{0u^TFl*_`(AnD$Z5~xzvq_WQw1uA&WoU^&o88!7o`*ik$Hd;wI}Zg`PT#&v4W#wV@+bux7m2*%TEUHFi3IOM3*)%OBn7<N?Fv*<3MiNtB_0$U010Lxb8rb+&mzGP+zJ{SSwyk(cC14%$(dLcldGghVAnLY{zDkGYYv9+f}#T^f5}uvRWs;k^IWo;mCGe0+GwA6jKDqr*ht1ODNpfhJ%!T7anu3GAR7zFwg%KZrqsUZJjTiZkhk-;M!Lo|tXS8BQICl(Y0h^k5liDZHDe%g|9>f2p+AyVtv7sKQ+k(}cY*kjX;SZmAeiJ`Em5f%{Sk!PwUC_fl_Mp{+rp2sgvtp`yjk>8?m!{Xb|BnmMB|WOz&=;hR<U31pWeemRj#D+S1k|K=arfy-_b|?Y`-+E0+PgELUe|lL+zN<;jniJ+m;IIKmpbo_2`}Yv#pCon#{dRKlQ>g4dkLpZYnOf`nTBp=};>&$x4n3MPBC=$~ejV$+4Z#71niJ%c|W8S1!!2fw1!!%;5_@U<(8zp+I<E<_cYEE*0vzfba`ebx-06Y~)rwt#sQj(q$BQ=T`+>ETP?kxq?1gWPa;xh$FGmMQu^%+Gf$*lGZQgHFi|-56HpWZv(QT=M2b_;O&1C65qV}TSSVyyduHP>mM?dYg2cupZ;BvB6|BzUdm5Nar5S{P@KnNN^tY$d!Jz3jQavSAT1*A4A}SR=Jj_=65hQ2K5@D*&$fR2^V{zL>&=@V$aQ5<*E`p`tz9FjN96MD-&6AAUw-<}&71E6axIO)L^rn6^&@$rdRm$=NIkZi=Q^v7UE>8Rkow)c{zpM#J_H9s3K@6?62s)ekI2Q%>;Dzxn-Y#ka%9XLitltue*b*`=*ih1ypzLc$0ujbGVDCH^@=<eeUMk=tmdU;8w}zvB^YEVxNR6c;v*;WE|FIqXt%ew3IBg=d;n~HJJHD!-h)xLu)Y^uS5(|H+j>Q=5xRD$Yf=xjeXlm#5qF4bUSJK&WyWj)DT9qE)_NEg+I<#8k;@wdAo*8Yx)y(kYK$yBSO9M#E``s|&Yq1VyOuYvzXg7w3{Rz<!GacQ?7sw;$K<QFN%=~MX*#&&?pZGmf9D;ZoE)E=f^2ck9}~;@8mf!DTFe@^P~AXPu@xGrDz;_PozOyRHH+pWa%V(VX?{U-=H4Gc<@}`w2Fev31Q`W@|KN!%AC(y}?$qVoDBQGkl^i9;KX>k^MiUUYBgWTNA)o)8n4OI&Pcblcv(nkj8nqf7vnYqGnfk#(I$IAS57jo|e!`ETj=QPc{XDry(p5rq!cN^_z|xGZHrHq+O@~+|#VwwvX}pPuxsWVND6m2k5|=4jdZOChiRo*?XvdRD88!-?h|Qt|^jz&8kWYEkto<e*yJWYKe6{P?m|3r@O$)4|ahifLk1l)p)oKgX9K)h`y_XJ1@-f-l#M0)9YJYyUEhIwcaGm7#>RwBKM@pO{gsL0_L_BH-U1yN<KpZKXs2FL{0!k2b3@+uB8MJ$UyuqrvOyiDLmFMyAUPlaKN$5?>)yO6^{BmzXA}Z|_ign%EQm}9i(8J)ut{Zmsz4nZ5j_fUaVon6Ux^8M~IyyOxmR?=%(w6a#j1tedb>6m`9LO#{-0B@^iew+}5rOt7XbV_2a`ODCceej2SL&0)gJY01I8qpVwtx1Sqs3%2)+%d!r^aEmk{a*bvBGZFC5O&DPS0)Yj$ARk<@#AibnXXNXtT%fx?@tLq|D~IUz@U?tVLt(sN!pT#q)g(L*-r7y2{fQ<jL{xj?cXPPY>aJK=%@dL4OgF`5iE5d&9}@)Tp1B&c)=7&{amUDqQlwYgWdd3h*1XMR~hh{z{d9y)eq%c0a5~)D@COAG$}4KY&mPTrLW$N}72jg*OXPBHajMe+Y>ffhH&syF<z4l*eaFUi%0glLbCw1D#6dM_@($Bu$`a8{^S4K=02mbMO;x0rR}%&l|uv7As1{n;6-;s9(3#FJS^+<EiQ1Qs?E{diQ4KK$GS?$D9d`3+PJqTE#g+YojTlDvkiqZk<NK*26O9qgWt7r&3F+AywH7FuOSQUF9d#4ri3~Wb~sPm)R6;?qmt!SoOP#oQ8vHf!W&S+mxN?h6;g1fiWP`>iF-?alOw`C4Xqz4c51<Xw7hldG}h=+>k4XUz58Ju&%Q!ym=)lw`R+@GsA`yASrAuB)_h!U|z9^Fm}I8g)VY8E9cPrD$H0LQs385`kBQgTey%}QH|s9yryAdg;G>)W?{xFGty>9w_Q;Ivi!s@3A|TwmB|yYmmsxJ0ZKJf$2e_=T^mZb%XW^43<j%N6&%8pctcdp$R|Y|L{vO4KT%Qi@YL-KcU3P@J(sEh%~w<%Bsbbz;A|p)xQ)93$rY$5=5R7A8+5gK>%dOA2rRUEjcMC(2|#$%lXVPRAxRHI+L{gNM!Mpbp#2(0>7DAF7_D#20f!FnuQI%&sz>Wt!@dF)r-y}Bvxi>iOmKmquE#TBUSCX@KZ6`M4Ds$;9ToC6KPRj<j?Gy(hqS0TD^EUc@~*L}%`*KGN?^P5$mTt4;2m{cosM^K)9!ko@hYxbZ@!hAqB7R`Sc{60to6{*6So!uBdhDO9PKQrn@W!*s&84{%XlY0GOTqFvOkb?!=k=Jgz$5t1U<Htt$6kH)_IDx&&&@f`h6GE6LVeNdwK7U{-D=QNu-@6Bhv#l9!{Ah7_KZ}ZoqZfqO8i(8^JWRGFTus-Ud53>f!TWk*vkiIqJ*Bb4rf2y<Ut942a9d|72!-#pE=bAUf=cir>|Kcs>R~pf6z;a^vE|C^X4c;*c1}+SRkaT)Y;m#J8W%Aw*7!XMDyEDGT6HBOb~1pqvvxI;yR7z;1cITw-Ia0HHX4M6*aEkgGcp`Is`_ovXIDMIl9m-!#-D)dMds(R(eC0RgPx?|MLgsuvXQ{f#TZA#a02)>du=S4qjX$Q`7o00{00#Tx#BtC+1C$!S%jtI~c11EhKl5@2*4tyBA)^43XOr@&~16L_t&NDC*svQ&>5pV@aK;d2@mSq{GPWVxb+tgB#&XL3Z<vQ;tQr{e%Ze%KdO@fsgiLEE4i6Y*IsFfwbMe6}$Nq2>Vmoaf=i@jTw-^gLdS^klamuNNyq*eqX+I|9?uejqGgL*u0x{2xa$)87',
 'crawler_snapshot.py': 'c-qZ<ZIc|wapreM@IN%cVDt<)J3G4oc;u2p2?Qw#K@#MFpza)t)wp-2XSa!&>0x)zVpog(AS>jPbmE0{mK{=r9VW$dSRq-W?9=H^gzxkL4E_&92qg6{?#ZmI>Z<PQ*;$CXGi@z;x~sC@GPClnvd*15_toTuBz^Vj^<bDqw<l}qY;qLLlSx{1gKKH<_9D)Xf=RR+Px4MMO-KB1kwwWYne7E>79_J#e2C}nxpU_pePo=a(_k<dFN#GL4+cRpou^q5M6+31L`9O$@<$(e^b!3w+nYyO9!s@hI+?^nE~M*SNf)ys&N{&;9!HBwF-nGoDLor5vMinzUDcYYbYnk@qtWYWI(g|Z9xjR$^`m?+OsDfnT*RXiKoK7nlVn#{lsD687VX8E6dOfFTqM(27t+5m3;6r>bf&u>PoregT@=a0^n^!86$V8*&=t+__r9LY&5!*k-$z%>Z}kt}bf<9<p>9-unn$A%#+#crZ{_LC{Fp|?zI17hM{9hq^B4J%MYB;lEq|8%%J*py<<E3@D=x~Pw2pa^49n*8qw=}n$p=1^Z|pHIX2}ppq{hR2Tx1g(B~O*^DBiu5CbJGb<nu{VxMj0+H!ZkArfXcMS6@md@$3BBLbCZ$lx68{U;w70`nF3*u_$8xI;?#~8z6v!zrJlv6<+3rHRdcDCzA+Bs0UR{M!IKvaWNQAM%1IPzI6SKtFK)P`ayVbDSY&iH!stpCcx`)9!J@5{|eDzoUQk|8z^)AwHIEyF}VEYm#*DdQaB{=k3Iq_2^Ph$)!usakpTZ?G5Rz!DBGQ-w_6%(fvLlGH_6j+ngL~6?GnT&*~9o+v`7_5f~DPjKia(Xlq%8Pj}N7W1p^Zj52$#HkWhoUofP|whU#rMoyW6QnC*sbn&YvQ@-)ccq?K_e2vrBdhQ^~JohHM<?okov+)^j72SaZ#nPor9*R<rsQl_^?N!C(7^8Sr%0eT0U%ZovJi+@U;BA(6ximTG#2D4}yGte$Q1y2U!uzPHXqD$D2i~zLNKIs<Id1%4u-p&%R3e{`wuk|g9=aXm{>;CGIswzbPGR}1OEj>)nEKtGjXfd7VEfmC%W;yZcC?6(C|4Y#%$1sVZ&x-zLyBp7lR$Jks7_U7YwykC-X*4QZtsn_ELM8H+wbV2N17fc&28+q)djM12iF6hem@gSi7<P$Cw5%~@z7e&t2;t3H7y!`&+OHJgC8OI<fllBxm2tH+H{x$yD#dFIupuyUlHQIp8w+PdSd--Iaf^nnfzdNC#$trsvS4`aOIN~<DkEU^!iP6A45GtBao*~84)`qY=8N4{7H;1xf}4w8FY4_)NhJdT-*$#(ir<_aU+A1Dkfu2`&xhT-$dY-h%@cV@6G?!&YD%~Z?Y1sc&cYmB1sS!l*_+i&7lo!jdfmlu=}$5aTJ&{22(b+(V-R+i78pZj^2-fx<K9OJHjS@FlSTXz7<SeQUt1L5=Rq_fbUPyU5k!H8^eo1?pk0P1cetOX_z?xO_;#?ihE}_J67zJC4P&Q^>XqgcW1{)DyRWPLf%cu2Qvk!5kGuJBA40)E?InaNdGJ_2*bb8sO)QWo+@X#@Gy((Z7Dw|q2SXktkU}7Tw5kxsNjw>~sCJtRFQ18*4(AZ2V(dc-T#$?)hK&MRAV_X#odmAMbe5Z8C!<`AOVMhyYlhX)W&7OwowoYN3>G!Jv&CcrAqxT<4Qf8@@-vt=goW9#XrWBI3ps3l1QfzNOu+i--O^FKn&i0%nxh2}ZHTQ*FVEE%%dT@<)O(Q_Tk?LKWZ>CNUmp+Ny#7iMEsFgVeDo+t@*rP~$H_1O2V^v#XDPNG^x)ax03=E~X)wk>kAmG8-|-_ZhWo_Xr9ITyjqr-d0}BZ5oDKK|_*OZGy!Q6UsU9r}YH{hVD%<K|sZ&hZB%vM7svr|jxRYRpAeGI)4gjJ<E8Nz21;U;ZJdpO+_KRYke_?BVefw+c%nU3RXa84Qg>iUw#-rCWk6GIaj#2%w>_o4C{trSWlmp^$*+lb(bu@ONy}h|pGD*9NTApoHb0>JRA8gbHnlGj;dojBQ)IYF;cJN%V(d+d*1F_3{paZHRZE)V;d2XQw8`D^pK|@QC)yz^vnnpl%41zu$y!7@Wnyk^T0roUWhpY}Mi71PSO0r~k!S4tes9dhTj=`U3%MuiOdjVAn^Gmu=m!!>G@lg&s%|PrjfOaS7NxQfA6Y2`a#`T<OGb5`YqAVf)pX=1Ij$fM@eXhEI%w|?@QUB)xEte?+`S(R#>0al5geAxE$ZTT}`VUg9H%&~bE4GMkG*ge`nTJ6vBiMFrVuq03I+{|UzqOW7YsV$!4d-K(@E(F}Xh2JdW)20IB>_D(x~yAll1vlYjh*SQtvE4j0!WL<H9QMQ-@DNqBK@d!O#8!@nJ{K}TLH7DP_@Bd>EWa@ELa$P=)rad8V`6aAp`!mhwaIlgSMyASn4(RTK9aL3+|wWv-Dk1OA&lO5GrhCYV)8KAbHa)fgy%c80A^*1R&!6B%1Dyq9EG}GTCMYd=X^@@%5B+4X6-NQDjU5=ut8n@OR+KHK=s}<tvcR_4iWF;%c<#@q5~mTik?xvy|7p0qG(cV;fAtq1M)t(V7BBv@bsasudp&CyP8eh})%<F%C@2v_(-{R#SOAQF2R@B!c}IG>TocC4$uR{jj4zV(|LqbD^5B_Krh#Ry)--J3fCtvsak4fK2bzC*_Li9-gvl7Uih6c#4}ipl{QCk!80GIKzVMu?6accLdT&Mao-S<1Omgz(48QJG0(2j@9tCf?kEwD;9oIsnK1`N#tp1ril^C{Onlsp}N#Ivi-{1v(olN-dA8)+Mm$kDX6lGm1?do&Dbzd@|miIO2=jYLz*BgBY3*Dpg}~P4%EC6RZMxAA*kZPkxu1vbSR&rL-(`Qh!IM`DyX#{>hji&XId_^AM;z0Met2M*pQU4LXynDEow_K9X(BzGQ3y<?WA$eT#d?|tsy84cr=I#;&o(0B3i(t<uSbpFJJxgB3s;m+4pc=8=;8KU^5*jdsJeWX7fc(?|A#7S8OK^?NO>_!#9XKLeZe{Je?fG*b_TuUB$k$W!aHL!E&I{TtN7ywy<KFT0$?6;f*B-V-myE^clYFbWt-@fD_)u-DMW6nd!jrHU5^)spk+!@XLrM{7P{OK?XtT+6cVE^-+a2LCdh)#RJSmWk+Bwc#th-dQY%;5!CgABwgf=@@p!zZ37+N5trWHafKu@wi>Qb;d1GO`MPdno%#a20qNj*eC`-^6%Re(5ufwwI{@M{29(;w3<~EWuJBiG2aT7eE5K>j8c^j_Cx~C09+4`bKPuH-f8}4R=uzQyHa}c_ah=r8X_B*Q)(x9_LDoW8N2A*VvUrh=6r1a2uNOSmcS?|AM>tzB`SnOfyWmxb0Lq<yJs{J@PQB7@3PVbty%pV2Oa^oEJ+x3o?as<3*Y-<aVw)uxy#3h?3exX12xU#vM}r^=X`VnOPY=;)*r&2+sM_zC#@WvU(*zk&!kzN{IGKSpsQunEd1}UGM+E#+%Mho5S%J%X3Xdnz-hfm;)PkhJRXxSPd9cbDjN%BZESaiuR$I2CTd)rZs>}R{SQ4#vN5GysY{eM{q?-1L&1-yf7(J|lSvmwF@t)J?9mfN?zKK~3gBSss)$|H_dq#cHT8T~ER$*Tw#PDkSrD48{nQUR+v4t`NN+eJ}G&i2Z`c~LB>*J<fqDhcI7o~ABo1Wm+7+cZ4UP=1D;UvoQfPF)XbY40f#&fbQXog9a*#UOAq!<jKL`}vlL4$#itF?e5vk%*|E0x0<$t6f+<_GKRVovI5*FaP-h-IpxPQSiHCbKwe?g_?4-Izp2{f#apD(qcwfdF?ebb|Bb7l6`{56oNI!a5!vBIQ8sGtBRR9AALk-?&6D<)hLs<jaK~zBB`Ztvov#>@G$ig8nl-3(6RSC9gwKOc+RQ=6FQ>#KV$jJp#Sg(t66k%RyHlRR)i$=8^V|%2!e4l;~6W=x8a#3R}Jg+FFE4lwH@`3D+>@XWrF1(*+D+yJUZ#r+HF9l+u=Jho9T+DOr;>^a?a02v5g4{+*@#FZAOGQLXpG$^dP=DxukL&j6}KI~!E_c4bf&o|cfTmnW_y2ko{Ku*E{u-pWdlv22?A%+i)gn(M<9#>OMDO4mQPu=d2r(x_;gVid^$hsc<igOucNs!I@pdaGHlk1>!FgK!EQ%TgNi8Zl1R3MkiVKnUN`HUM*$%=uz10a>_w#p@+2jqEKVmTya6^D7A4XjS_H!}G0NTAx8Y0<E}xu?whIL~th^>wo^?!4#2XM_jnxW;AiS#&49DQ11cM?f_UPdq5WNkxBfBZ7ep1u}Qofah30Af2OJpMXV5WaEr0IIy+7ma)=|?Be@55lcIM(=wvOb922W}Tqmp&L0PCPMc5XrB1gJ!NK5V0Dj~ih8o}D7Wvk=xP3Wv^m-iscfrQ_@j4oUr(&o<kHWqY!znD%s5O~-w0TAo+(fHZ7`@QZno%8GG`RP)4(yoPC)En1x^F@yN8A8$APe<i+X@^*=;zv?ZPs)^C`pcc^R7I3~GnVZoP>Kv05ZF@{woB(Im$n`5@(PkwR(Umth;ciuBL=etOC$;t<*i|P-`hJ5-_d<zAtQJid;Z0X7ush6jsc-I@w*X`D1o&bd_1^}#X1U#RJ+%RHN^)Mc&D`k#NyEl<!uajk=7we9*{c{g6$ms`9k}LH1{Uv%zT^Uw@6c@u(4z|PFs};?KmSw)fUEBU#5OvuZ)r>p6<1qJd!#>t((jT^4OyOY+u~j?(I<6(O_?q?naYF25wL0VwOkaSOW?>RfKSeSZHjJ((a;o37hWO{;$PKwVLNu3?GSp>ct4e276N6uUbWzHXMAnzR_;d*KuCT1xsvtkVA2r6_X<+nFr$3tJEg8M-sK>&4vXUP7Uxbdg@+k>8>(|RIeKi-(CQ7Yz7lgWObYZdIHGE*(+xtSmUYbPE*uK5vu}Lc{<i(dFH@$L~02b!u{)NK?1dtLG#>?uoHtp+>WsM!ntP|qN!^(ZGeSW8l_5@FX^tmjj=6+W{XWAp8nTR)t*RWB_!hGMxO64n~jwa_iXdAd!(mf2w`cI-kwzr8{CO}636ovb+P)jHd)4bM4NAA06tw=QBEoJ=qF+cPj_H?YB2J6=CR4BLs7CEI_%6YchX;jtzo)SgUt6it$@#t)G#qH`^PQv8;JZ`c5yw}RNg%!I9ICstQ<zB(IQBjv6*>^rqBu)j`N74sTg!9q~Nx(CgnQx4Q?Yn1r75y++WOy&lK<=OxT<Y`tcVXbJP@yi&K-qDv!CC)LKtnx^&^G_L3!1%{-~Xe;8Q1xiBfFT>kl`?%U?&PqAr|GmKn<<M34q*eUj-8C+=u?j$JfnM{8$WQM17r$R*<chSR(mzYi(nM$py#bDZJ9$6i+mvmgMv9^LOoAqT}aoGU=bVM15z-|z(X`tGEB@%N-8?FtassaNFyX=&rIH+hb<nsgFD{yvQMah}c3SUgvEGxAIEJbzb$Wt|~4c*Z^TsAA)_{0AYvh<o)L2PX#dS2XorW0K1U08-BirH9$)p`Rv2(FVVxpsLBN~NjpEyET6F0y1~`lqPxqNguuA!cKwrJ|qG-swGB+AHrN4C!T;o-x(adCQ0l{CQdbtueI9L3Dx}u%XH0Yv4PWjobXQy+ZFvYiU4hOT^hG@n*-fnIptm54dRyQ#ziHM%)_g1etgx313ePZu_LLj<2IjsKJ_ae>Yfk&0sGFrB=-cu)iovgtj7Q$zC#ZqFCrNsbu6#;ja`LzOHI@i^C$cq<Z4&ti?79PKHzVP4OoiEk4%JNQr}UfMfX7iH7i4_2*>k*x=w`2GiO}sN>Yz?d`3Nrz~d=91jT8=P>2tP*sNwKs}hKB{O8a;dZMoe%%@JJ<DT*IWg2M6Gcf#2*t{(q7WqZfk#Z?I{3{Zh-C|!r!Yxi!DAu{^jhg$+BOMRIwO(Su}dgeD9f;1{Z^W1S%hQ3HadRm^pnAcM^8QDZl-5j8)A46{qVZ7ka1V*@}!J)a0{_z{m{mS!BU~q9yy739;YmJ#aorQIqBhcWnp$oA3@Dl)-=D)i0A3N6_x{W1Ia8ahRgz!0>bX;ELbe~2oX+fiST@P$OBE07qqLpb`D4?F(|we9;}Y}Y0D^K^D2suilUvD_|Yhrq7MxIScpa%EK!`oI@A-RxauonNah?SHs^wxrcRpL(5qv}Xh#cV+<wD`vm5VmSm^2M+v!_nlZ2$G#<rW78QAD;UJTCjKb_Neu~Yxkx{SxhZdqT9RNUO648gXko+8L;`E7|@Ld&4(+lr~^>v^-|2KtVGbwzzyBN2cktgyZ6Y#CWt;PsDhZ3PF){Q~C;VXtmUye-7MZs~|%e1GU=Y3V?SwL-eBqtZjo$=4z+GZMYxseQ8QzzDY_#G`LVfwY%t3b(Z@RCa}Ryn-EgU2l|%o0MB3MGb|=dgD47M8`s>Cxh%5lzGzK`+C^ipmVBBJ8CJ$f}$Nb;KhWys0flNA#1<aJ`IcP@C=p?GLDy_07AH&4jaWw5i5<mp7eLT+gcH6)uu$Pp%UbUOTFGsM}Pv_fc@`%_VrQ$`0L4hIvBK1(cT)&0gaMkWiBA#%_$xthqgUKOvDbe@&!1bN1bMdM{)1V`KTUpg+xsVPv5XGhiauql?b5KP};_5fn&YmKq6W6ehmp$QuHqLAi`YeZPK3$^ylBvpQlyam&0#f(S*88w<z~bIZs`(bpf_r0@_(-KJZ4=EfGbQgz-061Y8-A%mG@k62U+nQy^51M}DCRb1AWLGNnyXaMm$NT0ahID-VJ)>*K?fVe_Je&yklc)Mp?enImr39Mj;Vo6kLlurJ21hEhxEMhgp;*hN%h{7T&XnQSh#8UD(RS6>Op4-Z@jQXM<OYEZlBm;?#%=tkwVeapyYODn1~{;l%RDDCliGcPogKs9*y^x(s^ioK+F5nj5DUVH0LIgD)|#nih3ioV!vx@lbKu?p%uR<KUOGZGSjCSvpyx%(?OZoD2CaP70vtEDJ7gM&{1#wo`4OBK?4s@MB}JLD3x&oE|Xt+B_alv0m6)SwdiwV|`?PfbeQ$^wDbs_|6^>bZe9_|UN?t}XHKjq0~QoQI*bjL<DsThl+7zTQya+bbYbs}c;vzzu98w;*2PjaL-Z6?SUcEL)90s~v8yDn1iWw*#$pWPnex+g{t)Ii0Ol@V_pWHSm{ZYs#SmB?geozHV)AVn%gps1ujp9I<@UYA}2>lZ=2cvU4fTdczIM9$RLie&1^i+FFV1Dn5N9H!bt)OQFiL??%I0>6r2bYzF7g2QA&bH5jE}nUZ6z$Ob`aSjeT_-MrLpba@N3Ki2X6aQ<x@(`i9rcEvKqvdCA&{=BY7tYo#^a*bwnV_A4c)0RtPx+gRqk9I+?>zYKf6TPZi7`*D6CumcyGShEVWW*#)9ay4Z3?rDl2Kz}N9@|=2ma&>j=ftMyguJlaq6(HFFHT5Sb2OG1!fF3Ko=d3D36cch$1E;j5wLe+9kQ{^9#uNC-FOx^zOZhc*Y^E^;Jar^rTL0^kY;dHn)KaJRln!*IZxywlaV%`tI*Tgxb0;)s>vo7iGsSVK<$*lJ*9&}2JbsU?Op7k?}O~!s#SRvLFSMv8|9nM+En<Ro`nW>H#d=24ayaVu^K;5J#_Vt;$O|Nr`D8v{7z2Dte?B+E-Io#1)s@+x_{wp0<;I&NGR0P+#n8G$5JfYlx)d#{-xDq6@ynxvgVYi)UIX+cxX(s98a8Mp#)JfGesetlAtU6(!8g+Y>t9eoXu)bKRN+%vB!yA(-TCxF+85_C|ZP%mB?BP`4x?C7|jN3WNcaK*u42unl-W{=MWv%%3Kbt9H4#E7*!tdOS?Iyyq<j|pGj6KY3n_`b!s_7;yT0|=t0-1ibT3QM8D#q<>In{Y#z00*LN(bR4!GyQ>jU`qB7}to)i7z7_F?=`zkv|^|0xnbniS}T?@Z6L=Q=awwLIxcTm3|Cz@Efv`Zu;D@%hDEbgK+ohUo;Z34CUX@8G2_B$9Yj^H-QzqF&<5xi<!S8w#*-ee!6b=lUuX2(l;OH9P|>gur)YOu_UyUcxicBkD^|D5ja%8t2aH`_zG*u=r6{cH+F+e>_v+m0r><2jL6jZ;}cVPEcR(58HkUUs3Tms!|ZRmYa-Va+@g$cm8uTLaCmU7Zl&mYjd}aM*y$rNGZRm~^2rZmY@IT{Z0~>lcM)_2|zg2Wioti%JjILw|0ikv|^1l0vdpel`=$1Njv(i(1p7GXfE^@xmv^M_!fhd@<{;B-Q;9TC6<<XOP?w75UT|<8>)ci*5Z-v5bPr;ejTE&1jaKrdTPnqD{8{v>4z~O__Hra~gX*Qe~7bZ)=~QWBHGFCVQO1*vR(SIUME18Ztm9V`W+{y~XX5x|;O)yS2<Y_=ogn@m!brsfcGd+eVS%r)q@!-HNwv$v}MjbSbK+TC1m`U@^%Jz4JxMN}LIkLulvcO{bDcb%I@eW>SkH1NY0oaea)4eOxrhyL`xGmjv%9OJbE)j<g9xLc>Op3@7m#^0iGPaFhLbk#TxwmCswS3g_o6;fvw%vA~dzYmJfv#)pK@Zf)@2P5yg<|6b(3m-z2f^tX_Zf}svJV2G5&CJs+*dGP?Y&H`5Aks~8M9!w6Wl)iFGqt9_L?|{w`L&lo06%YvfZ15W8ag1pRMhP_&Srv$<l6WtJS9wfp!oOu_r5<zvRwY;To;kD*twejWKk91jdIC3!KtnmExO|A5sRd6-aWr9~OU4=0YkHcc*5?DJL^2hIr}d(2K$Z(kfO`g8-EJ1vYSS8P>ag<3n)#rSV49ZJGo{+Dl#UFUln{HyU@8#Q5_5?<7va}k$|c}CZ{xIG7Hyd2f|(ZOQCU9r;7$;>o9A(riBm<?$TW>Q`F5?0zY<HX;MiG+VMpDE<#aG#?CPVO9KPJbv5(jy`RnArrBFM~j|8j3TqT<j_Siw5TaqjYM6M(z1kzH@CrV&MY@wuhfj}fqjdc&_9Fw!Yet!Fl$*6x$ZxIM)DbDR!iQVW>hpx=|UN-6}y{~b~d^d<o*TLaH-B{POR=`snukl!Fjp?YaQ`-`Ue;CMo>pcSv(g7A`r`iCZriha$WA`3`tss5swY_`q9t9u$;G=KO_HO3qxA4E#&C&5jXY*v;`uE~VyY+>~ZjRb+o@MDSLtde?j($<i;ttL-=OA^4woM^E;S;0vC#LI9{25XMGmca;yPp<z@AROGfU-CP+ORxZ&h>Z-K{Vi$0NIoeY%p3;20&~mw-b`ga!EvLMSdbrH`{{fnHjJ_Ke?V^eq`=kz^+EW9qi;2Wa@o|$i{qbc*KXVIPIU*;FTW5`4FTU%_x18=al!VT))1zb97a#Yzjq7)n>7VKd9r(l+j!1^lis*q6iWANhGzV(x(gtNCgwGQFggK%;cI-9mkVMTOJ-QEM7AK763wC3cF&}?#>r{pnZm6C{Pi}`u@djY8&Dc<uIP1L#=bcxlYN&*77+eBmW`;hs_aR`7eXT^G;PThtug+V{_3%w#IydTHF32PWg{(jh3@a$tYUf2hr4KM=iq{xYIUM&^gT^<#U=Fp5Gf+uLZAMz4zT~!M}g=^0n|RL70S|YGw#OdFCr-I@Qt28u46>G9;fPo{Vm2Ip`LsAm<%`lkT)^sRoUa)sblo1r#y1yUu1L<x9eMvu4@KiDUCbPdR={1$>iC+1d@Cc>TpMtqHDIk`O5Tv!Hn*q09vwQ>awqic?A6s#_!E^rPw{<)~uiDWrDTl!1rJ1kQ1*3nAThx%<*9zLSBMnlgbZby%I?q_3Q!bu`}bitxuF;~eGT8(E3?nCR6PA^MKW&47Z|&d(Y470z+j+D*D$C<_AH2a-I<ED%hpW{zKQie838N*mPMqXGXU9R|sm`Ljw5rv}F(<MJnFJwp}NCZP?!wNeU&j(?Mav<TZ<n@$Xzl+#gg!D$t?mgz#s0083SM6ClScGE>qlWY6+j7$fN@`&!6L}WXG{s+q^E&(t%*gz!!r&U@AWs{?mhR`GM&m{Qr3!4`<HWdm0o30#X3GSX53grkaxoClJVu6StZy_1<@xSex=Wd2OUubRLyuG&bWIL=<5^7#8r2}>=s`^`w+4~{dQM0@8)P<)n?wIICz7c~}i#_0A9JZYv=sP*kl0fh7VRxQW)@oxN7@PiZ=uj!IkQG`3U|D-_mu8Pz+uOZG{~WH-gT2UBizu}5bnik>xF}%?5`xVaIzm(HgESC0jWRhuZUt#s9oSlyJ)j0!Pmiw6I#>p;mc4msBkG&#&W;zkik$R&B~5QF=H7FDwGNlDGe)Jht`6#Iwp(_R*=?U4B`IHQF;F61Ka`AgeC3ShJL)9(3Of-eHXRY?Q2KCr^NV`V`IdF2Kt$W>9Jfx0f=aMx-Sr|GjM~6>c#I`F8;}|R8x1tmEO{Y-Ib5^tW0++j`cmVTE|;_>mN_LUQMoz&W<Zg!%xP8CB@p%}c~fgfGn9eG8C0^+$6a*-=)5R-tw{!=RB|P3clXkR?!nAf&siGgjb2|B2K=y*JSI_?Uh-`K>V6LkiR@SpUW09SHprO2T-PiCfMgQ2heCRxT|@~ntR#}e)M8r@<AAXv!i<GAVO^0W00`^L#A0iwg~FGmXKf9%3iar$Eh+U(4UZS3n$WIPI=pI;s!6$R+-j24-&pLVQq|Hy10_aii3FCBuWb!p%}F(+Nh!A~m-kqw>jOwsqnEab#vdg3Qx$qr+U2vk)$>FOWwe2t8tPZ>HR=S0@cJ5!2h{t6&2T9<l4P2EhJrViKCJ<D0Wjqw_t5&PdcPE`^oCGaW!6k7Sv;Xiw~8dncG>k|>dq=%Po^Y4^|!STQsY~m4HZ+F6xLh%C3)o4TH0zUQbtYp!uF1nu%{MKq6t{#i5aLn@DN!unNZXnld`?Zu#l?iSU0oz(#~$VY{^M<%=QZBwVf!JlU^84wXZ~%?^|QJfM$GM5*gR(*riQMFsb2)hf-eB>keo@meaO0oKtSr_;y%TuH4$C6P2`PPNhbvSbf=}eU&J>m0Kw%*iQRvUHa7#1(mqLP#<dO1eL9ieLP~upmAAdcVZ+&1D087_!cLRf_NX^TwJ)=>xC4AKcqi+@yvKUR@a^l^%)dnhU612Q*mFA^@Y_40)t8;K66I_An@GiI5lWkg|gQ1O*vf(XqoRva+gKff~O{(^Pqv`&J|+0dxe;s1~GRg=(2n=dsDlZ(Q_oKCukdRIC8<5u6db}{!}(iAM{HC$UR`5*V8(}8*Ek&_9=B*K-Zo=3w<HUb=nj5CnUQWn+zrhO<0gBz&i<fz_PBP7+M5t%y4>kH!b`zb+IO2-ycwWBIt57cPZG1+aG=N-cLUI!M&f|`}N1~ef+DB{^H(WeDwW$-@W(4kKg_HgB@*9Yyf?bxS$>0VHbo@1lEB7Z8zkf_6=x<t&Pnd&zPxUbz99K{Qb_wIE1E#_;SchN}Uc}<%ZF_vD#GK^ED9p&(yRL74zEbZ8U-qIV)e}bR~ubqZIaB(TrJ_&`!_L&b|;&@1h}|eJ^fPf0}!vB|@5L<-n=1BOoBQGPZpp-!lbf!T)xP2rS*@AOLolkOrGi(_69R)9<L8BPe^C`0x-c5_3$h<@FQX)BthNBK+hx-+A!!?>+eK?;iZ%yAOW({_p?wU1Bv4zV(~m-~F3UzWdjoeej)6zyFt?{_vmq?f33}`oq6}@V7tx^uymh_|`k0{ra~L{_9Ua``{<^?UNt<mrs85_n*D<!Gpj02L1l@fB)dqpa1y52mkunw}0l@?!({z_E!(y`I`qn{^^5X{)l1!w|5?V`!}Ed%a1<$^{*fN<+mRE^0)N%qwkYhJ(%s^|L_-c7}@XN{bsOx@1JJKb-2JQ_7Ps$zxNBg$nU*}1=+p-;rnSY#W#DUX))EH^@{NVmak++n5A53T)W*A=A?`l#e+_o4?Y)kFJ5YdLD{%eNs7r!QWvM0g;n^3TVEA|gwCwXht0N$hvF|9LBUuwm{GvQART5EEOf}jFH|%GZmQtvt5MA)$iR^k*<)Oh9F>e<0MukfebK?jrQRvL$d=P9vOpEQlILm;LKi9*Vh%T;s49pW4HIjdY+g-<5Zlvnah2{UK|B+#XPm$cliiu-=^DFz`KeHk&pmc?8;vF7JJ0cp=kGiR1@L*<0>pCDIPR2P-^6(c-xqTrVq}&@9nP6k2u#K~X~VCzzOaQelXq}V@eT<<NYSp&58io;QYPd6%#`@=9jY>8TbYb|bmtH6{f|HV-A{i1zutN9iywhs$nQ`7_Gh1b=l?wT_OI>~6w?>n+Q0W}Vs!WK{uQY5{)g`+cl4`!|AfnuLG}M|6mZ#g=$SsJ%OXZ;=|JX!rBK?x$fl@5T&>xM-;?gDI!YJp`rlt5v;Q4VHA0LNuW<()-JF8uh-Vx^@K$sXDYnhP5RP>V`rVyB{Oo(5eejJ>-~Gm&VKhf5Yvis-^OC3`;Qvtvx11x;F)T0lQc3ko;Ztv!N&u8&dRoe~w^V&itOq)g8Q0%(avDifS^So|a=_6Pj;r$RmWgMq2-{l<Hr@XNVl*b`S=Cmer3fWHY#FJdJY-FnB^+|g-t8%##Acscn(tPda2TZxiFmA-k;?mZBx0U7$?65bC#49lVz6P~#F=2Ef>4>0GPc5%mjcU$GIVbk6$<6RkDFhqnl#DGY)FVfVu)CTmM3~~O_>r@D&6X;sOs`A8qa+gl!<^ZP3b#VvbY88d$B{Ec|+HPxqgSv92iJ+$N<?jEErBY4U<V@pss6PF`Y)`DiL!{OkcNaWm(V<R$dMDm3z<^i_VcJuHU&try5P`%NtO4NpKxIAk_=ZZu|4Ql4jXeG%H;}^LTJw#~^Y{Ddq%-!~vu0`{4HM_8^%dVAd%nK@I}ufEZ#t(32*qx%hU43vKk6Y~@*I!*HTnlzlxa*&U!nTb5^D<uHfIg5pL$EICN+g$vJ<zG83S92V&#xW0a3_XOAC;>AfQxgOd!uh5|B0s=9<_JvKXe5lZ5M`^s|ZTW>-71m(V<<q+g`&;qGZhc9Pd$pGL3U9M{opQ?=*_(9L?aCEtJIimSRF^t3f5Og$jc{b{c94TOsX9_lj2kxsY*p??<4XeM#(;Kt35L9(P1U_(uzCxLo71j-t%$mOrs0;5%KbA97l5dnXZ%-s5QGyQ+bbK3>=NWKQ%^0<w#Cb+@0y`!b7@LI8=5^==6Y}v{|dh1A>_omDP<h|>Py$(xcb^P+S`<#>$%KsFyQ9_Un)p#>WlJJ2kNHQ(3;aq??OuYKZwd35InWjK^tAhIka{DyxumQyG(pg=0{MmXM=L6`4Mh#Rll?oL4{wDuE<fDs*L{W1qu~O-15isE%E8V&{T#GNDJhxH;(#NDM(OrM|>l^3Z-@ZToN46K9SXzJzeNb$u)9&3ln8|Up;N4`dU2}dP{PqZ%maM1(p&aGjGK4$aOtJhtCFdd0T{C<u)Bz2-Ge?5n|YYBrJ^L!w^e0qv$I`w-$?2l7#Mrq3hdhT(n$~oIFm&*>ECb%%7Lh$n~=9Wf6N#UyFl|QBsVLBC=UFS?j_>>WAaHhkC&mGnCW6>Sc%QY1h@8?mEn)gV?>$u&m>W$EVT_*BZ*VhrQMCN;y))B9>il$jy20HYCwZR!aJ0Wzbr7=i|~&QCc~%FFM`v_*Z0s&3$p{ZH;8k_2U;efgby?lp_heRB5|+%EL+so6Bwc2)@A`dD`BxOA0>eF#Os*<s1NYgKw{S9notq9mc~2sHa8M>1-68_F#DOL&ZR$q^*)JI6WevFW=RjFLtLCd##lnIu%*fc5Donj#rLg^1ay|0nQmwa1m+D2T>#9u0(Kj!LqcDEueOukKp$OLRSS3)UL)+6SK{@sU=9?wddx_?<a!IOJu$71-g^;i^HH~$$WR1*RSU@BN&i-03OK$ead7@k3FRGm7~SP+~KENFbbaz!=f?NuIv-P2KE7xLfD+vy#{$w#W_;GOe1qieE1hxUpOovC1Eegl+*^U%GniKmX?Z`+SlGtmy96Z_5jH+bJFP{Bk?=hl~sy_gz|E5US)(p0kftN2T0*w8cqQj$Zd@f{WDy9z~e6ic5TIxkp{d@onz)45TS*_yv=X`Ul={Pt3vg#L5v*V0J$Og9S##9_?jK^RgFvv@a}-P!&**r`OXt<Ba^(x4ZXKoJy$cdw{G@(Aa3%)X4MxcQ5e(wU|7Ag!hbGQmE@V=$PDQioxe*-_?Pdmygaz}!Yfx^Ci?ImJ(h!-&F3@~7rYL>0g+Pp;AK(mhH6s|_)H=DxqvRuLMLUWD|fdCO8?;aohj|Wo|Baxt|>Dz9?pu@GV@YFtPJ6*nvIUUd*!t&FArY&k2r#1{l50XzvcO;5Ai68C!^CcnKz-~nNaa;*!AW$KX&=^R-Cd(Z(g&Gs!$<{%&e)6O!9^=87^jvNoAc9Udasq6bh)^8Sg9ayH#G>X_ac}3Yv#%e?0vBM9lwx+VSnYBG;M=#Hu%W&zmG#zT&G<q@6cOaGA3dbI*0yrQljN!54B{QmguMmdm)=fV;|V+*M(9YuR1b)r+#VCDavWovX9e)z}YrEB5%@l{Cv1bCz?VQ_5VOoU4yR#hkFNqP%OW!<i07saeFOTeb;(wYj}P2Eyf+bQ8iEIa|JI4+tS*rX)|TSy}5)aZ<G1$ekT+?-Rpf6D->f=@=pf0Ea*4e8`AxKb-1@P7_&&j2NP+=?XBJ9jjWIGzELRI<+Q*DRZu<#%LpTt_Xo<t~+C&>?!F`Zko<HK)!_^5@yEAGKbly7VT5t98EVOS9q|ov%-+jCld%f&~0)e2-T{zK!dK1s{^*8;&7ZvQJyW$ahSVljtBS)4ow!O*D1Ty_7n#p&v6I0#sO1Fzwp%o7D1^)<V0+Wp9SY0b8ybrfSIjw5VDd|diQ8RQFRrrtDNe%BO#qHCd%y2mpJky%f__LSA#3J8%4g!vN}O>c{AdxCKTeUXG^6cACzfEb{D6$?OGKXNO`ti8rU*}3Cq#YVHxv-Pl21;&*`VpwnUSn8<(0RF)gxHr(BxJ+M0>wAj=p)1#uOovodf?8)Sx}wS2qZHY}4`;dY$t?JIUhy*k{sr*sFYWALWLN8lfRo2lI?r4)^HzyzlR>1<}%30E{9BzJPLpO=J=u3Apt9S_%NS8W+f?zpe5H=3sR+B@o$mxw=ArXFj=p0~}zLs>cP<Wmc1yX7Qy1a3Z;K72rJ45P-&QFa_$;iJ!*-+17N1OIfSl7grA%sBB^fhk2#hiYy~V`^UsO$m|@R5fP60X43$Ci*qzU+y?mWwnL)8gW7l>K#_V+&BdCAxP|F+lRh>G{Cn(xdW>0POrQ0bx%Ew5}cWBP$2_@FCadfEb3r1`41d;0Au)qZ0Woh+R&xRhR?Xdkg}X7w98{%o*wXy&Y@R0a5PGynM8;1NjjJ@`CUu6g_bawbF2t;{>&vcEmUcn6L1dWWYX$Ba|!H?19}BG@qvw<RhxjW1XLOr`qQlC+0xD>f&#f6$Ph+Oh$QEnAsm^erUzB!>7>c`s&Q87uU$E$C4A7teO`~zz-Tc{D9}iCD}X;32mb=PY7v;mY8uWObJTq>hY2eXKVSZ=q;a6!Puu|&%_QfD&B0y<YP2+G3Pc)lB8lXS>;7rLCdbE-;*y<8TE2{p+%<!_Yc!UNZcmeJYKQItF45^39!9#+-W3DKr_NCK;RL|w5=zu&%6gLu(F_>L2kDj+$GIiWW97R`IQ59!1#Yy!G8)K=uhMM$r0u`WTQyiUz~Q5gE`Th|haV_}L$0;fdo1h>`JhMFAa3d}u=vi{0O5XUmSBj^VHY02#=jdA)r#kCBoGxQ#hLsBav>5Yw3R<4B&%{A8Bg_~K=^p@^8F8gSKurK0)3T{NYZ_MHF+UPU%h&rPfEZfb#5)VvVZ^XJ2U9M7M7mSO=Hi3fbPA^F^B(9$ahBRlV^kBy?>$Eynpxa@IiYQjMxDK#<}V&P*Z%R07MMCjuAU{0Jg~P{Q~jHhYPF8a%Z6e_wW8QM>zj@M!r!Lo2$5IJpi*QG<4z0{SW_f790W{CKO6(_e6Lc^<#zxl3=tzBfIzS{%6=%>V-unXd3H{`*;5*+PM3l+-`!Qkii7!6yE3g<C!;58}_@Mt?uR%4eeapzxUs#G;zO`Odxt9WLL##4(@;Wb11RN{ktEi#SDgo7gt{l_U_;P!LSZDWU%kgN|VCm-n(UTD=h#1$G?i`GT<YQKmA>B3&6ig-JrMk@BSkY1QDv;;AIu5I!B@R6O39FWB&_Op3pa-*8%(E_qfu%_vYm!RjhUj&ln?Qo)ORf^MwwV1&(0FAkN?m9eM=b3=tiOz7oh3ZgWCyQGL%bHB=|zPVgV%UCsh{hgS2i0hN~SPVg!>bSL;qNgh<h#$Eirtmp0y5!;<$ZEY>!zgx@y2`zQj7BK7sDMaa-ZH~{~362j-&V3T7-?oOM-<24p-;^4Jd1V=M^s7=g!kcq8zoB#2)QfFCYxel#m<CKdC(O9_ULtHJ{PREmbJ@-<=b*uQS7_epuw!sm^f%k8YUo=YwMJ<03Yd_hG2C<X@vrXReSfyMTDz4aaQ=3U9jP9k86tVfD_3v)+2Hz1ufKNvMt8~^l{tf|PEgg%f=%KNMcR)^Om@xgNOKz^5{F&FmFQTpx9jr0&>_Di-y`R-FH@pws@2_4n!0tV(SqbN6ZtGVUeOOhIL75rbt~5rGs>Dei5c0-q{Yt@7{QPR1F>G5%0W4gE1qP_fuj7ijt>R?+ACSiIiNWVM*@wMQ%i-D@XdD_r61y)1sn=crZrY+<tgc?95%Z8V(A1{uR1kPjjS=eK<KT{frF9ahtE0SGumrMv_5+wW0CX%CaTETo4xKK>HbO~ie=f)%@3m>=1bf$G$GSl7!6{cGS1Op#R_nKE5rv#_CUK=odzG4eF|WIQ6!Tt<;|i3<@g9>M^pxSu1ug7Fja?WJvqi!Ri~(pbaFu1=Q}0#IWey0?ZAl>?0cdF8#zpsvFP9Bx>?I5*{8jML)Ij4Hl<pvPT;27H@0}r;Z8O!G1709PAH>|v@924ZXI@Q#HE$JgeIty(-Wy<)uKV!V-T-gySq0@cU$3k!GiiJYS;5dVSbJiLx!uY`Vzj)o(c90+2^*s@$w#b8)h{JHxcCTdJkux;%-ydgqqRnZj|1&$_+f~ute1@NB40$PPm$qXdjfY$9syG&J&9!4m#;FwOsXvndG+OEIegQs(4qCqK8qCza*hd<F{<S(Zm6m=_`=+S#XX@EHkBUoh48D=XVyOeoWtgE~2?y!x;(R`Tu1{7d#%}fBU%<ng;q1xfML)f2dpwr*+Ajd!Zrk{jzf;%?NSonUYT6cBWq{)1LD#?smu>=Pl97Znx|d2S?1Qy}5FBV_C$Y6)Z>sAt_d715^igIqj;{y`1f_L@XC!QjSd8q9Gji2kh(}7@y8A`3JdJHyYu@%3@dj%BdDIQ&Cc>h;KR6r$lWQG-v<SE>~6ca6TC$i86B_a?T$k=}{|OTT^^9G{-Rdl(s2a=LC)ek=y|R@$p475ISPf(5HHXNNz(XJ1L`JGAqi4=}=E|t(fzo*J~N;XT45BX;Ps#IraAA$-Ez4rEFX|A~qq5W_eMH+~IO)>D&Vh=bi+Z0TXaKOO+P98Dtj2UAYEfFVvu}(;GP#M#+O0d}MgMnDEA|j^3^ba|{5Rg6$+TF{zPdQ%Kw~kSWR$9XuQM1rwIw3R?m{pGB5HY5zJmub4W%B9#yx7unUi&DdcMwD{V)&^-SvelD-AOs#5lUR3WWG#O!zxEaA}`LsslxYC+Fk<>8AOG`#6kVvpP1q>ZkIvosM`XDEt{9vA00gU)g*657XLv#iq_G?fvt*Kvv!geU+5<FJS`{59|-l&4Myg9W6oyKH^(pRIPGj~$;L90?3XE&0)g#RJvd8<_B+_QOMNwFK8`hmoyN)b(+T%hMEU=fl6V#hVAuEa`em)6$JMMP7|hOi6A9F7_nV5QH>RarA4o4tr$h1S@*G-^At6$8@eL|FhD3^;wpKnp+=^P3kVQvGS8r^8dg+PXrh)pFX=AARKi0y&M5`~',
 'diagnostics_snapshot.py': 'c-qZe-H#l{aepVk|Dk<CFgw)V9*?n{VDAH<NeX7H4~ja)h%blH>}>Du=w@fu^Ko~p%K{1XVB{e{h{OmCLx^ZWP-Mr59VH0Dd&t8n{y#2%Nq$w`J<~HgyLXfYClth;>5uB_s_Lrhsv2%=Y&=M%pG)D3Z0bi*S0tlR6vk2vLw_76Ssn)BhmW6#QJPFKC`kR&NTfyF6=57i#W0M=V&<oL=yP+{+t}E+eJh;Kk~9~7I!0e6Z{OnSK@vqW$ipPggdXuADdJqFT``m+zlic-806*jAUn}Bhd?n8r_xL_?_EKEf0)FwoHWU&QTY-k)g%8z;h;B_xeow-+4Vt|#CoBbpHDExtn&n~<*-@g=c6Q@mJQiFE1y3MXQMCz8uTAFBS6-nN02)Nvnx*hGzPIUw}1PVz@I#w>nHk?%DhNp#-tY|{xIWNJqD7?v%DkYAR%J!IYmC&{A0&;)ns`V$XPBrZ@>E}O%o9EPyMKn{Ok%pqkd}{gGA4t&15|6jGS{`C-$du(HG|!y>Q$mw9XeT`-)a<EF6hA$(fQpc@}1Q)^V42dU&Lo(&RJ~d*$nCm`~Ky1PCtgnzoeXz+q~E1XDy@Oa~-KEW2SWBpPH2G3lt`U9oLrp%JCgJq!qEoo4SckL*8$wjrX$G=k^4%TqL$D&8!!#{RROP3LN_O@3*@5@r>*aD=?UJeOIA+Ph*D%4k@!N|6TS50l=zph7%;{B|dnr?ibd#{fP}LmW|z0<hjg9MB(dLrGYMNvLIe3Vat^%Tqt|`K<3bArwg}>)?9)Gl4JxlI|fs0!?BcVQb5+H3wat^J6*htJ5+prn9VbLX^e2aaN?#^Rplf_g?p-OuEG8Fbm@>_v1jCf#6h@L+fBSt}7%43hrs=M=51kwyOqeBzr(;3IV58&`GU|28%jZ6s>w=ne*(9+hfs|9VOVCSvV&87`8#Hff-&_Os9T2?<56qjg!;m!zNE;oYMiajwMEmE#XATICJPBj8BrlXSLho5sn4N2m(|%pF^U*#Rm-^yQ9ql{f@&;IEj0E!r^(2@Z(|CkAs`?YI+Qt4Ckn(9MpQtTH)Lo_*s}aec=?bf8vLcKZqpobudd(^Y<s0{|1v2Tz)CAL6J|cE<O&$IP_x?TwQ$b6L>jDl4CKwy7*bRBaYRCteA_`7x~r2vp{_H?CRpD^!!Vap+TG`=`qC1Sy<EnF%*`Eo7)EYl8~nJ&?2rE4$UrSX);b_)<)1Zd@{3EF{z=uN`lu>V4_z|8Y5_tufbl-YB2XEYXKT9XK^=^L81_#4kGBGBO`y7$)TtF4C&}7w$AK8{>5f#`Fmup!z>|#?B^ZL4L^5|T(_4exgQzGM!Udz%Z%7+=uCJJvL&}z4V-Hs2$Jb6lDQmWhMKA<P_}#dB+R^O_h8svbDV;S92U{i+_JsY+2nfdS!->TI|gD^Gx5N5?2Ee!)3?~|u}@D$x9S|J^*ug?7Pj6gygiR7NvEsM*1pBlt_72cjE#fJePK}?07hXPX4o7$aQLDQ@!3UtlgoXGQUQym-@D=uU3-O!xPI<MKJYfo!OXG~a4SmoD{9-2iZseRXd(}av|!bt7G{dy5#PB>OS9lpLrKrtb`_2c1>i~$+u#z-x*h@?&83}B{j&}huS8_C+L&8H-mV}s@5|9)#tM{HYG2bR$^%=|O=MD2mGu0ZKx@qCx4*u+_%zqP{rKv|=dp+<S1&%xca(aZB$uCH4z-9OCPIy-m%oUW0uHZU{8y~j>4`@N_eB8Me=VZRPXxjJS)k#^<DR4Ow7khd5k_Ek8cf0ysqCD7F&55l!%-oRoW?=8_|c43gGop1RXPH%9dkkG6?tGh5Up1&zjNxx;Yg{6%5m2U+VrtrZn2#{aj(?`tFBf0T%OHv8)7AK4h0Ka0t3<)4Hd0mEX+X<l9Y~3Dt}Y}CAEcaxeb~}o+3|7=&*JSE_$lt_Qp{%=s0)OcHFvePGf2`X!!WmSZ&GGl!hM*iO+HDWNow>sm%!44w_Cj$dG~dY>lBK=ZbIbl>-h{uc8wy2n{%DtbEM{=kdd>_ny8X;xQyTgop6rmqqCz<d?sSC*tJtlcX-xj(gqa=={RhB|frWYV88=uJIgcvz<CD+Dey|75xm8sUMPrkvfF|)6IF9D<23dH!m`Jn&kP+t6qOp_)(b8%a+C>rh63mV;&z&{L~BR>P>I#tOt3L@(bNgoEFX9t?B>f5=Q@XAB4?6^@#4~Ih*+RUi%Jj2CJ2?4InZe<KjW1{2+kNw{~dcy{)N6j0VVZ5bw)74&z~MZn@n7-8i7=EBL0RL2J{jXIAUEizSITG%P*hGgZm9ZgxrL+r_&+t_8B*f@UKv5aGgXFJd<7kbZF6VGKjeSPw)ESqB9SRu_GSpcRhUO(_aTZV&zv1xeI(skVw4<ThO!OY504trvFd0RZ+}m505G1PxtbIt|=&hkYNL4P9~Pjylr}zWqp><Eup9&v7H1<t?XFNsPOwu7+w`q^Jq%UgS>)L!T5sTLFcF+i*wWpk+0SQNLX<NG*x9D~iu8`~BKxDZgKX(dxLikTEMz9Ah?A%hwrRuy%vcJ7(Q&lnWd+p=PNu4UMJ7(y5=((53rM+f=2^+*+R{iY~cI90Jgr7|cC&oXA6{qKzFLf?SS4C?Z)xQ$jc_7pzpa<H96mGUBrga=yAJwwJ>iid=pB6cy?Me;x4Vl*7tduB~Q`K+vJHn&Q9;m+WKsNKh~~1RZrCA+!9-v5YyMrrrOTVjt!8i(Gv~e?2m@=!6Dfi);I(;m~9I;c?tJgU^P6rE8_Vf~6tu5{P654zGn!8{yVHdd;3b>p8_>!o4D8-_n45gVBdbcj&j)ON+VtXt3<Kb0k#FY22j11f_QAYG4aur^dE+cD2ZqTf?c&?VBvLUOui}(9(4`S<P0BYN;41|0Oo2PJd$wYHZU^)Wm(HuGc9NvT(KKwC2On3=mINyCONz?X|H8M`RSWBPB#;j>h6Fby&sLjHJ;_ZM(H0mO25D&efCNu_Cb|m+I&mRZ_YB<#ZFJbQd;s6=VeH>!XeAGT};K8ZgN`*2FqM@Q*H^@5XQ*Cy;$y248$Zo^*2g94Km6dxq9eL*XNRgoKTsO>oma-6GczZ*y^W`8?no=&xc1fNQ6@Ly8zKc_@ZO{W{5NmgBDEfc##(>aXip*Znqbs5xQF%Ut7$w>sZi)NbsGyEPiVN<{qTz470NL++Xgtju#VVOEYn%XeFrfU=oG9Phn*dcHam6y{Zv4$W!Nb~rha=}9Ol+HhzcfZcCFQRMTP<TT{|F8z%NAQD&`C*j8aE~Op?Nl7O2y$ufiqm@N*|ABo}R4`_DOLf_Vbau0Rj<!KEoS%<y7wvDqHanwSB+mF|5$*t8Fs^^^+q1JBO5|)#WH_E+Snst(PT}<P0Thi)Hz^%UlGQg4JE!3=pY*ry-u;6GZvFXCkL+*L0!b7O#Vf<%aA)QZ$)NTB2rG~fZQ>Y=<9;MZ`3~rUl(8S_HW=dt;1Ib+QF6L@n)<W8AJ0!GGL;K(>HKs8<+Pc>qnCZW^N59}`gu^K&}IEu5|Vyj>~5*O+w>SYyOZrU;)(zH%fA=r#DSj70)NKgC3Z_(0K%LGl-a(~TK)R8hO6T)dg6OlMC{yl7#3USmS?fpL89VcS&3bItOgnfV$s{(nl%_Z+ka%dWsaYH_3ZLfR)EjvqEse3!nnyi20BO_hgTOLO%asi{Kjuo+yV-c6Ze!UK`P{@xr*g+%tITSEs%)^e*}kkP>f6cO!$5QV7$u#MG!H3R%ER>fAzPQKc6!$nbG<n%=6F$9KU*|(L-ZGDD6-X@n4X8y}I~u!T8b0k5trUMqBtSKoblB;r5ddScQtE$U=Vg%vT74C@ttr?(N%w&()z#NWi`QT?pL1BVH9YyDg2O&ipkB!^M0Dn541=mz1&Pb<gBfL`;}1{9gP6{nF+Ug#DPYsn6!43f++6|H=O2htLHRemqD4gyL0`%jaR)`p$#$@c{{&UZZS8*h@Wc0is#yXz7BY{Tl0n!ud**J2JQ8CHJ(u`fAs0%c(lzFND+k0O$HTe+<;^I6H#4q&>((ZA_2cdg4mye-%J*G)g*3Xx*NS<~Fx}he%A&9gPoEK|b8CX#iSSWqpCu#)eA@1Y;bgT$v75l+xlrK;62nKs7{8D;`p};_eXkAdIs8U9Ir;4{#Emfw{+*zYu@&?%QvX9KZNnorKS!{`4><xZRrEDw9X9ni9s{trGecTca|TK(5@BBGx0vab6KEx)U1EC`%63zj<}>kK+AuBLwb!+>Z-Bn&XhYx%l#nSZfm!bsXcLU?X5^+`D~iQ@pQn=dknp{aTC_WjwY}cf~sox}2Uxi}7x)%<UGZ8Lr7)s6|bT1*HpE(^UTG$xw@b6Cx(86)l~`i42a}+AQfpFi(NhM?{0<Q%3Q4BFt7QKi~@{oN@AD2Ovx=#-zUVK}Rm->cxMMdZX<Z<o7D5+Ae6QI!+x-e*0^`ftVmxtWvFJ(>k!FfHel|UcxyiZ|Hjm2Tz1LtQ7s$x`|Upj~^O<ItZvC29tsZ;dc(0v6yhGhxunslkrKICNbTc4A-B`^9hoQ0x11hVf0spn83_%<Q*4-kP=IZZaJWe6S5Lof~a6(3qVRcqm4pW-6W{&gln+~QdPc|X`3;;{7=wVi6Wclj9`&i5p8_&S_`|j6fp>(-3`%r9za4a|7l7b#HECIiAg&w`U<>P%NuDzCW9nzth;@eApgdQ8h+KNUVerxO&I~YPTEt%o8fmAL?hA!(=6B}i}M-L^`A9e2bYkKK>hV49Dx`A9`7_7l?@L5`wK7=YM`Vsop6M9>V}t>roshUmePO$<x_@|q!@)Qp+t%*BF0UO|3X;UM_zOq+Yv#*4Qj)H>GnwTunWT(MST#zF<(YJ#S8_4_pkI-Jih!nb`Sneh%-QJ=k_gy6YFE0B2)949D&H!?`rQuu_eUON)6q<HSiN?s*Asy5}Q8N&LOb<(%<5QZv?LcvQ6TuwO^T|Vi|^}{ZjcywQ3gLU`-$P6!|Jb5F?ZeQl1>$X}gdWrRrC6Vxh^E(dv~>!a|lu+`q9)+;95Cw{O8BDI=v5yFC<;lKz+;Lj`rzOSX5Aa<vy-a=ql3U(Ll$Qaz>$Um6fpRLG%8lPzY$Ms~WFPiMwVTiS|3pU#X`6~%DoSr3*kqiP7m19cTBI?Q9wJa8NoH<@F9%+a6o#lp_Za=S_y%;rAal~eob%UAN$$5DqWPJMcv&8Nj|IO3iNDO1ZWTP{ts@IS1OI6Bmil$u$tfi)ZJ=uuWuMb|90%z2{5zLn&!Q%M6CG`>7CRgR~=7cK$S<+QyYhO^fxV^p#gq#&wM%$3SQsJ^fq`}m3X@X_mU+&_5qkPAfzK&}?R=O8{es!~Go>zt>L?mv9<QEy6d2;~{oYxA6DLFW>LnqNfB`tw-(lBUOfS&+7~1c=JFds!fs)*+HrrJ$yYKv#oa2gmYwYH)P>jp0~6P7RI;kgtf`xi{Eo<?u9isydcH-!NC#T)A|FtkRJ(=`u+&Ny#Nmmc#yQts~YZ@YIk+E96(NOTRS|dF5(H-yAARiw|42PqD>8F~n62ov@sD@YlU>>z_v9HGeq(D@isRAvWCJ8gyQpGFwxv(x`R8^Jt^m-GYzP#@fQl%GK4kHIo&tF28r~a58wYF73Q3X<o12T*a^zW#P+}Cv{SAoEP-C5fzq!1b!yj@-;VXZuz=Vza1@y$n}6m>rr=gRkcRaGa>;HI=;fe>Sptzos!k=d$cC1tKk=9;fGS}?%86vg08kc=~%#^*fz{nK@)RpiH`0HUV)g0OS>bxN6t#i7XtOV*Oj2M<8bCd;K?o!G}BJjTB2BS+k+$OtS=a@sx4N1-N5J46mppKm8s%@-Ay7?>#f*1*DX@S6llVgfxG49NmYjCpE?!@l_k~VRc%OrwX_(3bmlR|hEL&&3Tn?s*=CZ3XRDAbB}W6q@w2tk8XPQ+*aEtmvlWLmWtK&>(w)~VT$fb;v}<Y%vn>i?neDjip;)SVbFT%Z^|Ng{^^ghZ+;G*eNwk&6Vj1r8*(f)v&qrHr$xYT-vxdp~BAEpl%)+7M1h6syNjh&?P#e**o(aIO_i(HTOvA;|J(Z6`rSW`cNUZ$k^<Gq4*a86T44Ojxnt=Grx~a`DQGHFT_jg;DzY?Q+A78-q1=urjj^1p}9w7yinvE)~)NHjPO!clc;)-uP!Xzczh`RQHY=Tsg2@m0xxPe-mR;*cj+F7gCXRCOfa;@nVW%?YBwvkrhl-V*ZSJfGF@M^it_z-!;qyB0Xe`mw3(bge0LcFZ}T`LJqc~_!=Y<xTl$7N%d6loxdZG6`>&BoP9C5eb`UMB$Wym$YN#|MArJ$>}#?WYIo)@h`!e&=)UzxVL*L4Bb4xT$$a*R5F2q010d1(&f<aM2R&I=KG>VJpL!YXr_Zyy~1ZSfpEVoyfTa+_X2&Ty54m!jWf%`}=%JBUU8`%bKgITfjFtGgXvugSAYFMMDIN0-xyMPB043Oev4)`oW{`dXJtyef#OVEF0@LqKhj|h&qV4g~e3c5$6~&uCaIGC*H6~0&e|($o^Nv_obXsb^#f<a7=YPLe;0#YP_gFBPkhBTuRb6Q#n3pR}o=tcC+5ftEReuwV=)QV@ZTk2qk|dLCqD*;=9#U3Q+m&#R`(tsH+#c;iavW^6XT9ZLPY3N!4BGT%m}-ZuM=&#?37RfACn^hx!*NFDp!q@5P;Tu;Lr>B_B+Ekef+)Y(B|qBiN=*FW0>p!Pu+e>`2WDb<Ls_ujtso%IGG#5K9hZ<Fh6%T#M`=X*HvWQRIU?7&o*Q4L4sOE45O6c14<`q$&Rs4JpUV)j!wJyPuA^+VKhZbcV<<;}d!QnA@XLNY8WasdNj#H(Y8?$JyKjqaF2)670>;Mrg$Zz1<a)Fp|PldM^La?DEUbs$jYPIT?-V>F>wrlb{++VHL2p<!GKe28(`ee4{+LEC{p8uWMz(<p9%kv~cyO-H>XJ6A_-r-`9E`F~aklzWEA`*mC>U{{nz7<Uj',
 'export_snapshot.py': 'c-oCtOK;;g5WXAme_-7MDa2NSpt}fM_)wrJ(4yD|Xto#EAQ*8Jb62mB)Fy6_|K1srlI?gmtqwNDdGec?kDAS9*H2w*JyWb`>rNGZ&VK!Q(OGa{znJD8w{6$EAK7Psj#+r<!FlGbD!?(*;0`Kicg`Gi$|?`lE}zY2(}}60Uwe=H4#thGa@)$>jqjga+e{~=ZEMykPa>8iH~2c8jEdb(S=&A`<xuhUZjM1PH6Nr8WHp^=C^0>yC55O2dC!*93B%`m=Df{aR~cWK2As&4s$9#$j$<FsE-x;x&Oe@ATwJ}wQ#zx8eNmOs?6*Jv_&~-44elTsd35<`{5G5*Z#!rNG)1dXX~lbAF0Ob+nV}u1I^{r*1>f6-ttrF2YL#|^^2u1)T1G2^I9f4NhMk;(cOu*D4_PbIh~a1fZOc|8^7&8pzOSM2H$fsaxB@+R(>5z9bz4Z89h&A!YpIfr;EP4udp-}PHx~5j*7lGLP|IPYWEcL(65)(;v_vu0QP6gd?@8j?-WGtug94HEIpeY6a#a-2dC++f@?-~39Td2Wmd|$0jpmyrvjHv}unK-(qB0`Zw0?+1h*Dvahm3u<3L8<J9acpd2kbAZ?lm@-75uvH{4R|GInq5gN`c*u!{UbAN&}T9JOt%6XS}x+{gtLsl}s-lI}XAEo(NX1TPh!|a0AE6x0YVI{;o1^3s?$N+vgS^Zq=itW+PQvyREu>_kQqu@H%E1DsA+bEe;Lkx_<r)JA^R52QP3LTMrk6W<1f$A-oo$OOVY;ZB?oIPAg{7N2~{RW`N_kCI|RpT~PO-$AR)Y>Adp2;~T06AF9p?Whx#j2BbDa-!G4`IIY!(wpk2r$P0wpMV=ZCpR!0?lt;M4K314s5LuRQ;Ypi&<UA1?CaDYnZ~ilzThtD5SZ!Rbe6byJaK#pC6~-rzYq8#Yqgu>KWE%he2MmtsQdL$65~9qY!ayj-Ch=12^^O!zi$B^$3Pb&U$dkfH(I%;Qbyq=5_m-A9aklJsl!2gbnx`YJ;@?0j5BZ}tm_o1@;djoCXFZWO_A6aNc=tFWC;pG~W5r&vLn4L2HG(WS&p)+Dgn>(SUw2N}IfKUa7Fo#^#;krpl1FkqWBl)i?;Dho&q(zZWv4*_;wU=!Lmt>F9^~sM$a3Efm}DahB^T5`80t-?xk>k!ZTm(qsO3)@$CTe@&4kPU3Po4h?{5XdLi80e_%pIp5$*%Wmw_}BWY~(*?>fL;9vYA+XBa*i2cP;*rVk1n>?NZM<do48f->YIDIfU-tietkZ5R*7dt0S66;PJXFE4u@YE&JrQgMva{Iq@th-u|%JQKp?B8v@hxAQRGaeJWFDfVVfx|@JoP)(STKf)|=$MQA(1o<K1yd1YagfBH&tHzWl4`(<Yfy34hDb4ZF8q2o2{C0nLNCaOoC1wa;UnP^04kTsZ6e;Pxl9DH+;>X8y@-L$=LTC',
 'priority_probe_snapshot.py': 'c-qZ9O^@70c6S5%4~%>uTkdZ4jI52^ay!tD#w%gHk|B+31hWHz-OcW%hfTKmF{ANd&N=6pTYw<99CFBMeaOK<{)hib-h1`xi!8EdBpZmZ@HEM)S5@z;-dEM$zkmPjHcC@(SI^UAG1^7z*ozj6xGcRzzTHJdQs&vzn-}?I85iCrj+SvzdP(NJiGQr)vMPsOzN?Zvi&F1xoW@yFZ^!rV-@o_Cs>rthBwN%)5ogtSRabQpL*MN#FDmb3Qvk4U^E^F%A1~@EFNR)JhB#^(SMgE<sN(lkn#^U#tWHyYd+!r|xBe;FsfQn6%IaYoRU7q?mkf0WZw-*&nvKeOzAJJ>fqr$Re^<#iW+-)$0+2Casf;Bkc=tn|WJCP1j8{dpRtrhfc!8KwD1h%#omR_a0V}!p$#=(Z-oAYO%A0xqu1NACsjfyalX*P4c;w&v<i+u`=g<N^U*G%0gMa=rAk?n%r=E|<c(|#y=`h;ufRGW6^YA^s`Rsl3?KXY-<7_hiV))?U1N!yEA6gJc-$dEE23A1OKW#=wuZBNuxKErly1d)13t)=T%d7-47V$_j26jHcW;eLFd!N8+y)3RS^Wr>=iy|+=^CVjaiz2#AV<3L~eleH|pv$<5k`z{2Rz-mC$7y~U7r{V$PF7x0CRtfU*&@d8Lyw`13m$w7Z)4)ia{Tt~H^(@hgpU7;7??5h{IX2NWc&{TeUxVz5cLFjBCjjg01MF~tVEL!8|Dp2O3xT5NbLcl>?$BhWo)kU!lUQF&!TM%qL`;O!WZiwdj1+3E|*1;t#WS#UE(Fa6~DYH_o&N7><?U%SY~AtlQ6Mu4x*ftj0FH_ObF)VFVegO;QSpX<6S1z2G)Ut?O^S((W4u^&@mLCjL{9C04%O<4N0{AAWHR6$tI9vScyF;EQ<|j<~Uo&A*$tob??xd=gTWOfK<q!6walPOuY*t_W2Ngz>>MmIH}@o84PYjPihSP5ZdYVU~>4m_rSwHXST5MMCI^fV3NRJtVY|UO4d;&c*Iu{>dioXshRs44fqcxHTS=)k}8eA`k^i>FY>ne7B&YW4MA{p*u@1bXPabek5mlkMeiamk`-&T-p)K)R8=sHV1*y+q+m{@UIUHt0yK79Zo(3@KQs<jakbc>a;jnX3xNumA}#E3&w;U{0z_jIv%Hv$yo0UHyWGHQZ-*RJT;K{`oSb~?F?djKfIzWZB1mX<k>oW}lLsEli7+u1d48V6zyq4@veTHq!Qb6gFc^Bctpw^TcVIZc-xbV8@Xgdi!LLBo%GN{gd>~jz=nKO36%u+87Zv`E>S}|3s{+Uv#@U50`3iQd_cTS=7Nadr73PHl+4JFMt$B)G?dc<sfsLjyNxVeOGENoN(tHu65_fXN0_B~k<wuFAalKmtae^k^oQ-&E9(oTRT>I~U)T3v}DzF1;zPV|DkmO<z{Mcy3Sdxt-(i)8jWvY!pU}L_BKD8tRmqg_F3N%HO)-h6BvW8R_h9)bSO&IFX=xNB%Wt3DQzy&p06<2zHSF#je06KucfWtJ&VlewaIJY|`StX1NbdqWom@{0Z%YeF_`e0Z2XG4;=ums`Cs`P4hQos~Y;AN*xURG8QUT>CP0%%ml<87YhRh}ga$pJWV__&SU2ZumOIW6xqt?{F5nQzB+mY_lkCPQ!hc%W7`&cF%_ql#1{E@J3S2HptA)P@TU@|ib2@V=TU6o^I!rg`Rs!4QxNTLWmUGc*m4K&>qE%k1GiTAb&r71~X4yevU?ZKEVZ9m5-wH?Nm#wUHUR{Oc$!WAWA+h$sgtHqVwN4OZkeSZMfq=sn^!5bnw}j(7CI-X)L@rZ%N{03$5&V(H+7GD2e?Jczn7RqW8?y>iG)0KppCx?L@(+@W;^R&rjIGg{ct6XS);q?q~Qcj+5y`|TzIY<lGD@Ki_+fV2@>2uZZgfPIoh2{x3_?f_#4dKnldo`dvgl|7HwN!H0xioP>$_!%!qA`RyzRii`(YzhTztH4Mz$#-?NK#eX9j@juMKEV25?eyeA4(z`$G-815aRngU=1crQNCV{!i(k-&xN0EOivgct+R;Fv@Da%v3CS<<o#Qesu6-N@oq%1OErBh&;RDt}pg!2Jd59)OC_GReL@G}9Bhf1$OtD~poO-{D<^~ZlL=-tDM1=`T0j#qprlM460NG8WE11<QFthFNEo4mF$h1f7fG+v{lnF|bE3qKj?Q$R-sA7S^K^vs7V3nRb!oTD*bq(JhQQ8pJ9t;+Om;vnA%NQ(ZV8ZL0K?l$%DH}*aBuh1{P<@~^E`KXQunq=}xdM`#l4<JQY0pF)$OwGtl!Ouj+CXnCbql!ML|_zwhMC7O|JWnt>btOhA^7>4IC&~L9L+1SE_lbk=p8mOHNs6Bq#}$I`m-_k{vd#2GjvRX)lH`kv&aFPF1Ywke(vX2v~+82w+bi5<Wts6IeJ+N4+j(WN%)U@6S&Jil9-t(Js%B{_o|uNvm|fK<d1HF3tGM;Gt;qjDTm(cw<3B0(J}1SfnZG<Xya}|93T}RIh!*XUpbvVo=nb$p7LbDn~utIR$c945r7Fp3>t*tZSbX|JF|TO9~YA)5KBzE7>NxQAl(jkjJCDXlJ}8hac#piRD{G`LOVS1cBkG07LVpB;u)J7p#cVR%TQGB;?yW&S@Ox<hxQRUE3{XnzWlPSCGqn|o9Ax4?Dw0yd-Jlnvk1TFDH0NdVc7)$!@)_`)k5e*i}3|>N?uZLe6p(;nKSik(202=QWO$t*y{b*l1jQ|-gg-OcipEfvGa!%M`Z?OSqJ|fqtg7c4POTsTKo*a>IKE}$45~aW8gOfcPF`spQ9aXd(52*Xj4_YP;3h`xjU%sm0`><IMEx-+M|ttL@F#MLK&1f$f^-)Hr%|?OranW{KgnfOp#E+|0r5j;;|X-g6ATFE!}0w0JcI*F97B+tpJP>T|QLX@xXiNt?pk-3pN4T-h|f@#mi(3tpW=0-Hg{i-EV`JO`|H`f-ye7s^T)BfiUa{(B=3;N{Wmzp8{wt3Fy$<qI#T72K%TZq0HouG2~`;d6~Q?M+9D$o9OWIA7-wpcL8r|HbHix%jc6PL+|s$FUW}p{`9_7pTG3}Fqt^@wgYIt0-SaV^6#P*Z=;qzP7BN-^nc#)J1lMIk}?*75Jg~Sg>{cuI8InNd<<^jb{v9%I+JEupUiy}-}FOn>IQ`*`@IVF7}PZw?a>6>hewk`{Ldr&&)?&Jo(za{EM$BmHG_-96zTz_0I@jZr>$0oa;hPm4f-r~_T#mwe(j@fwj6>ghetUmJR5X{58{VQ8)>Imc6PLJvZ;#U;}+1DiD+RP4Wq+`2tR^n49`hG4wQcC{TYHv6mFp)IqhA-Pe9rvV82{|y@h5D#q>qYMg(h;a(pN6nQ>2WNa7?r0+NboQL(m^?M#V58$mkKf>8t%5|>UxU)l2=GulO%Ztk~SG!-C;McK5OdS%)z9cmHDqTBB?;|()uCAjUHxh&M#9i+DA6AI=u8vD*2vde6tuF_^%`NRea68QLm|5T@d^uWlJuDJj~8pAL1#Rn6+Pt5{c3?WItKh{wSQ^ePUAq&CG=i$&eY=|0nTcy!Dl=>hLBQ%;e6h1Dq8F%UlUT`2UO~9IwljscnwEazc&AcpVGL}0_)2Td|prxq+BowXY=9DRH2UMngcs96FhB!hIb_rG3#W!Vv5(Cykl_cJBeSQs^b4TK)SW~QX<zKh`<D02>4KHqdd-XPG*i(~o<Bah(9NSr^Q%eD5bZKRHl;K>0aKs_jz}RM?iitZc&@nc@FGau(>5F$ds9(l7yP6`jt2zy(X{&7}3W5muz!PO&Lv$G<_<Q5>=%zzT7(am>e&rRlC<owdF9fL$-@ivA;;BlvZsNuH4t8JI9DrU|9><z>fL=_7<{&L{#K@Ry84%_4QZzB%tBj58JVRKxiJ>Cp_JB)&8$mh<FhEFzM0iq$E0~;>ITJ$p9cIc{7~jZ6GN3J{!{_M115+j?rr=<6TY>3^+qc4mHP~3n@HQ&WDVdKmeCYWSgP}b)UeXdX;nfud9>wS(z$Y`Jka$3ENKg>opPrV2Q2>)1bwAVPB;3xGgzz=R0gQgQQOFfDAJdW>$;Rk{qLPxY(;SLih^<8*Tdh#B3-+Kb8M4=ZH)}YKa@o>z960r6ABkXhfvsiv6oVSTrdfPRbP-B5FRsR>UT5i{IjPmAu*x71MAFy*Ga}9Bf&YLx=$4Gbg$!lyREiiAD?qpN1Z+#hC+3b$&vf=zkk25bF*^IPn%AY&cIps|C@3i8131V_Bo>pBX0HjZxkI*0RFykx^9K%Ta|iFG!dwIJ1F4I|8|3H#v5Yl`p`B>DJHxzu{_x$KZ@g_%LKWg-WAR6GR&2}DEDTEl^^MuAZ3DV2ELKu%gj?g7jsrQ{kfJ8@{5K^I`gv04YE`LR9`+3ZQd8d=oTGuDN-bbx#1IqY2&Im?Jv{)F;AM#Lg7lT33nYy|Y%IEvmZiu^4(kr|Brqy8_bCbwdNPi-jo~?=A-;{VBrj;DFDOf8trJKoTwm|2R<gq^XE42F<PZ~+!lCxh?uL#{RBj%Y$xm@7dp4XtlZjeBH)Zwg%Cz5tk3RLj6_rz9<3nmz26kjJ5o!|ir(O;oV{wU#Rfa<vgW4^<CFtvv3L=>Prk6++c2Fv%Qj>uwWSO@7StH0#O@^e$T~UD*xwuKXM0HASJKQ~cVCvF9Mw~iF#!7*XL?^Xi`bYumicUds@fzbgmUt3?z0zrU*inHQPXHqJw7WxKS(o$akaVfg#CXj{je1MByXh1J#Yj!%(9{a%zlMg8Q{Sf_>}V*DdnbL-z#Hlg38+hd1P!KOT_K`f4dtR?q3xt8DfP?8;&_k7*&E~*#BYYRCHUUZc3UpT^xw8g;ud@AJM;pSK9>IexNo0y+h1E=zQd1eTJ?Lau!bjR6#9uIi1gVQk`Pv4U_R#2d|fVz7*aeNtR!)172X<l`7WS}Kun+xY-`|m$cTJMEXUl$RA5`!LAI^e6%^<V^gMVVYq*5j9by=FW_nd!JX6Gc<iuU~)fZIn25E7N0I<5>WTJ;I{&BFQC5r69+%?nG$=Y`hPH5kz$FPttB$d}m|4$Jo!RQLRM<UaLZ`fSH{EkhA?l6n+M~7lKJ9xtGpQ`A`&U+&!5GldrE#^2L54mFHV5~Au<?NAFEa_wbS(FkSPN3)KkH-U9OF27ug0G;;H3YYDhO#<z*HOAFE=AFxIP+Di?(M)1tSVzB<5rMrJ;z9c2+B=We8!B+Sf^Jgx^^*;br}%lK|#F$f8~Jm8Ts`A<4WUaj|2U^Xm6$pOBH@WsY1H~2aDI33-a}xaK~lD(k6L`)4_?Fb{bm2d8$knJbwlT3r74ZT8vo1%$pcXc+jYK-LnEdWFnTpRcCSmVj%h=*J3UqXjHA3MYo`+)U@?`O!p_drv1hXUu<yxx-svxVwd?V<<9?#5sm)!=(<ZC2E%PS!K6htLJD1RR?S>X8@(hev<6Xk`K34M9r!u5a-a@}&Ej>`L?26ySt&&A1*`JcCd0~3U)nW_?diyd8VOByL~uB$nv<+w)*Ip1;!VJ%XsA56OjPL)`2%;Xz!qp609?6&BUA{x2A1bQAde3!3d<nbRnkP%<%t*o4bk8WLJEe4p=+b^xp`lI`9{O>e9doGovv#H>7Y(MRq5-SRnS<U{JYda`!-%-9IGT354vWnWZgBE(3oBEQ`dO%c-d4$8Ks^=t4<sPn5trWSzcP|`cmKppV$Omb=5`syDK=FM}gw2a?I=i{dVAr;du!DARl#*R~oSLRI9)mV?c(@WsJ~rO>|K279db1OOEZ8vu_!QM?gHL#I3`--4=S!?1pM;@)VQER8U0rPSMy~n@m<Tr!NkB`&g<Sxs+j2`6Nu|yT(!7G&)k5o!&62BLON?K$czDtHMnjE2{9X(<yggBCR1Ix1-~1`tWu@9-)hf48%-pxDPj380tLTBwLA-DrP~DnARymKXa}e%}>!S`~i{;)ZV1Tr`~Ja%NDfT@B$#_2DDwn;$l27Bf3wi5`(P{Mk44hpc81-nyTJ63D;|2wFaFfv;9zwADTw%;KU_Wz5Z5bhi*}@+%SLbN6Wl5vNkQLQ1-5a!PPg|Z!a5NL2G1HKncBNz1@{i|HA~HA~H0&LDzyj@k=3DzN!JDm;7|LchQsw5WO6Efgv5?ElRTiz5tC@!=^j9fjo6qLIr3Tm5U@1{sPvDrFFS6bMA&KgvPVrIzDZOVKiO;US}ct2O?dKwl+HpVmmbMd;m{u=(s$Bj4$an@m~tY;W1qE*bd)90csb&jN!HiD252MXbc5Yiu14ZQ!#3(qDE(CIEOE+0FV=#&&62S-MrW$;3+iqpa%5WjJsm=6m(3mpJ7yIG@|5hGSHyRpib#MQh*iG?J3CAJMG$$Tl<h5NNA;r+k45a;Ha(Z%&bJf3pGL+wBi8*<wvt83cfBFdJ@~X1HdEO=isHl5lVf_0h~Wc>74T_Dezo#wAh5q8eo=$TrFMF3FtKS_qa9LIh^Xuyg$Bq{r%hHH{r?g+mrAwub&@V9tBPYD`9XR?OOw+Z7|2^1e)Q6zzYZ_p|+r7QDUJtl48g+cd#qe_D_R|)2r?*jm*rsv|KxTli{&cNgi+J-6Gv-$PW@go%Q2R^gD#zC4TSIO#>5aZVH!38}ekW8DKnIHQ7U8vO=j%+gL*MI`>#rF@r5MpAvM45Y~9p0qfXc3z0I|uu7xA7Je33LzFDdI|g!5xCf?WO}ETIr-unOR3)*cP^MTpDNeBRSboXZ4bYIfCPw$OnFK2%kVWL&jFq=*DY+4VAVq|Fp$rhhsn4H)b&TJJ&BkY+w5$xF^z({;VeN72x)wKsiKFg#WLpd(xJ0~d9yCPuFk5Y7@MV_FfUm@HiLpcsBrjsHgGBnBkJpP*PmCjrKF=kRr~6A41XMIC4Or@ellDx_W<=_jOTq7wL5Mxo4MTwhrOO$p8g+0wIm2GMq2e|C)MjkK6_U5*3!A%f>rqWlM+axx>KI6GRxp=xWVI+rT@5-Pb}5zpkO%*{F3{^NSkOgTT+!W#!*rA$J8olXl$!pLb%b4Vs)lpt4aCJKbh@lW2@KFmf{=eaBEVB@09k`*>b@~YBBOa&#~e(yY1J5aPD<Rs63qd{(|H<Y=e~^THR?!|H8-_*&uo=N7kKCe4`p3O1>P;-$o(`Hqr1q6gmw@e@XosNLUBK(IC1eO#V`MnZM<cD_3O|7RC(u{U;Ybxwe;7&{ri>oKmYu{|M~0B|F-b5Uw{6W1lqut`7i&PK+*WGfBWxh+_-lK%IcQr)euhG3`)Trv@s#KQ-CT^+DP1!0q!5Y1+xUr&o~r;FggW>;ID~ZQ>BT=^bEcUY<0$4su}qQ`+1ze4Kl?EH1wvh0jLUU1A?dWvs}O~k2nq0duZi1F7h!8`UC1{v#xz14@Zd_cjmZ~t8|akt?uZdxnpZ_pNw~v*B)HO;B9KCQ&cT%bwe?Ce2eF{4<1k2rR;6uaZF1)je=MWpYZ`^B=p5uL(jbaqIJH9PRo_@f;&)NP>pMx>{Pef31F(o*wy(mo-_B9^$vTD>qPedKAu0PZ{NdYlveCLpzXdzLf-x$NA>$Imf*F@z_9bagGPLLO!0ZDlkP~I$~DH7kR}t^f%(wxL(0cgVa_H1aUDp`!)%7xtZlFuu)C(rmPwO{?vvMjO3k>sP8_#%PNk_lF2S+|ce|`0kv`~OjKwE`<ZqLM2XIYIm83yiqwQaC$4Z6H6xYN+O?)DctJ#NhTcz%Ne2f>($0>xtYi)0`1wELh3)1LmY2C$g7cRC-i0HCi_cb;>focU|RGkXziC&!X1O7;s_P!&QXIdI(bP<rlZtrD3z0>La_{=ct^_W~O$k2#3T<Zu`A>gBc6<4iBPyBsl6iU25xQybrOz_*J|E<oDV&REewDR^<O-e@zNp#KH>}I^n*2aQ)aoBZ$cW2?oyFG`%Lm(bX3V#em8Ge|bH{)4gBT3Z;*?nUo0gAi<e9$3A&2k#j7Vu=C=XO;t`r6N@9>wb*yD!+r$%r3P)Z(b#*mFurtx|>}`5@n{%lNmC2RSaCkr0J_Oy^_qS6{K}^^Ln2b?<>XL3+l3JU9!A?*~z9-5XamkIdZ{r{Fpr8RHb){dOdouOxu~8E5S{VkNO5=ZNGeTh+fN-Wn~Xd*>SDBaccc%l-e+IVt1vkN$&Aq6UbU(C|zfevzvRf7*A=n!H5Q83Lm)WUv3!=_+>lO6_Wmy@vp-Q~0g!k8gdH01oRLmjUukUbe`qeV+=0j9}NBn2g}a)91_7)zd3LIbM0EE{Mf4HcY{y&mr#|R@J6D*rjKIM~5jwC68p<F?dNMY|*BW?4kS*-RyQ}GO(XKkfHl_AIH#@P<rwbeSr}&9XewR-#vRBK7aT6$k+7_rP>t~Xo*`Yh0??D>mgN&N;i4$FovhW0Bqj(k6yfc79PEL_K&_DNN%2SQRu)xhPpgXlwkb%#p_pp3e~U!_wwv|J1NrL<#sxK$far`j1$gb^DM4Z`I>fFW>8UQo7=MkeB&tAJ8Txh8T|eXumHAW3URDF#mhyg$|=R$*N<oh^4hwmIOiRed=>Z-1B8%ACcjegKBob!vyaZ&I?~3fTCP#8q)OtTT&Xv`Awy=Z8oGAbl1g|OMGs{wH6*!0R#Vv3$g14AJ27M)J8LE01sk$XAp6a_#;kF<pXsnm-(30a9qkOqUhvPDF8y7oyW!u8N8Mc(UClJ?u^{|w@x`<Rx$zjT=nkP2o)mrHk4BPZ;I6Gp{}%1xI&M4~JzB;WqjIwy)fAt<MMvSDj<AG(c5wJ)GW=pPX`C6-g4D1_x?p05k&rw*;P1RH3d6ldT@gj!VJ?pj{aX%_<)ac*+&q8ZI^txAc?U4@F_iZNv1f1fbl^@$JfJm*>KeU0KZdu3-)@aDl0~$O<_Q_z{yNXsQ*bXKgtLYdP%Fx2Sgt0Ntv&nZ8{f<u<Z#nic99f$cIv-94v(H4y*LiPeRlFfg(Fa~*yBO@8cW87u_2`68;5ne!JfSQ%kk@XC*m&JgTsk$UJop|eI^+lqCD$RD6X0IjW|Zm><&%*{)=CBc+4g|zl0k_!Eq9%kuYVa#{XBt_~^~?vy<cS)$8#4m#?0`{{E)LH#5aIVl%O_E=nadB{AS-&G@hU|GOw>{P(vBd9^|sGA9*hbcaVHNz`3~rYu51G>R*81=_WZX4O<zc1ranc@giPHuDhQam!W}W`8)GjoKWncjzQ;SZxxW*`jX}+=34q`X)lLyuk|Q`qpK}Q){m_5!pl=(5~jhkgQCsGbmy@51{@G>ZaF?h`@eVNxHw+x>m<E8k$hz2b7BL^O(eKjr~`G?q`wKTG?ZB1v}iqR7eY0)hVsYlvpHn?^-R?b>VJbW!+saGu?pA041G9{(Ikzv-@tE)kPF{N<vshrNzN_xR=zjL!DWSMr7ZC3U&8X>&87c&H35{x>8S9&rqTXjel{E1eGV0?u+r1&F^rt4Z{=F^q%|L9Jih|ZYIY&+_~!Ayvlvhdx=kja^;Lsqp&CvrZY@Q@uz-(zfbt$_VWE9=LGLo%joLus7{j;qUuGvG+389Hj7)W;>KoB@z(W?UGWOD#IYkUey1wO1_O$dD4hLkU1;0Tp8p34W>0('}
EMBEDDED_SHA256 = {'audit_snapshot.py': '4d08e3ea1613e4248d35ebfb40716a5fafbf1a6916bfe35a99a74686d8093bc7',
 'corpus_runner_snapshot.py': '595b4eafd72d6192611b3b1c2a099ac562e9c07f6d78b346a03cf33e5b8de681',
 'crawler_snapshot.py': '4f3e6e74821404f0215be3157a7259b5012e827e54fca31921ee4ec078612ed7',
 'diagnostics_snapshot.py': '61305fbcfa4b8ac0467da54c98bba64235a6082358facdcfc58351f8336afaf6',
 'export_snapshot.py': 'ac461b827b046ed7ea8fc4488a85d62392e4a95920ca9be7e8d714d3737a4023',
 'priority_probe_snapshot.py': '7cb7f7900c8900e4bbcda855408ff94b5e978918cf0fd56b34079b9a217167d5'}


RUN_DIR.mkdir(parents=True, exist_ok=True)
for name, encoded in EMBEDDED_FILES.items():
    target = RUN_DIR / name
    if not target.exists():
        body = zlib.decompress(base64.b85decode(encoded))
        if hashlib.sha256(body).hexdigest() != EMBEDDED_SHA256[name]:
            raise ValueError(f'Snapshot bị hỏng: {name}')
        target.write_bytes(body)
CRAWLER = RUN_DIR / 'crawler_snapshot.py'
CORPUS_RUNNER = RUN_DIR / 'corpus_runner_snapshot.py'
AUDITOR = RUN_DIR / 'audit_snapshot.py'
EXPORTER = RUN_DIR / 'export_snapshot.py'
# Diagnostics chỉ đọc output, có thể cập nhật riêng mà không đổi crawler/hash resume.
diagnostic_body = zlib.decompress(base64.b85decode(EMBEDDED_FILES['diagnostics_snapshot.py']))
if hashlib.sha256(diagnostic_body).hexdigest() != EMBEDDED_SHA256['diagnostics_snapshot.py']:
    raise ValueError('Snapshot diagnostics bị hỏng.')
DIAGNOSTICS = RUN_DIR / 'diagnostics_snapshot.py'
DIAGNOSTICS.write_bytes(diagnostic_body)
diagnostics_archive = WORKDIR / f'{RUN_DIR.name}_diagnostics.zip'
from contextlib import contextmanager
import traceback
LOG_DIR = RUN_DIR / 'logs'
LOG_DIR.mkdir(exist_ok=True)

def refresh_diagnostics(strict=False):
    try:
        result = subprocess.run(
            [sys.executable, '-u', str(DIAGNOSTICS), '--out', str(RUN_DIR), '--archive', str(diagnostics_archive)],
            check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace',
        )
        with (LOG_DIR / 'diagnostics.log').open('a', encoding='utf-8') as stream:
            stream.write(result.stdout)
    except (OSError, subprocess.CalledProcessError) as exc:
        with (LOG_DIR / 'diagnostics.log').open('a', encoding='utf-8') as stream:
            stream.write(str(exc) + '\n' + str(getattr(exc, 'output', '') or '') + '\n')
        if strict:
            raise
        print(f'Tạo ZIP chẩn đoán thất bại: {exc}', flush=True)

@contextmanager
def diagnostic_step(name):
    try:
        yield
    except BaseException:
        with (LOG_DIR / 'notebook_errors.log').open('a', encoding='utf-8') as stream:
            stream.write(f'Bước: {name}\n' + traceback.format_exc() + '\n')
        raise
    finally:
        refresh_diagnostics()
if (RUN_DIR / 'crawl_config.json').exists():
    saved_config = json.loads((RUN_DIR / 'crawl_config.json').read_text(encoding='utf-8'))
    if hashlib.sha256(CRAWLER.read_bytes()).hexdigest() != saved_config['script_sha256']:
        raise ValueError('Snapshot không khớp hash của run cũ; không được ép resume.')
if not requirements_path.exists():
    requirements_path.write_text(''.join(f'{k}=={v}\n' for k, v in versions.items()), encoding='utf-8')
print('Đã chuẩn bị snapshot và ghi phiên bản dependencies.')
refresh_diagnostics()


Đã chuẩn bị snapshot và ghi phiên bản dependencies.


## 4. Tải hoặc đọc danh sách URL

Nguồn được cố định theo revision và kiểm tra SHA-256. Chỉ đọc corpus URL;
không dùng query làm nội dung hay nhãn. File Input có thể ở vị trí khác nhưng
được copy vào đường dẫn ổn định trong `/kaggle/working` để dùng lại khi resume.


In [4]:
with diagnostic_step('source'):
    import requests

    def sha256_file(path):
        digest = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
        return digest.hexdigest()

    SOURCE_PATH.parent.mkdir(parents=True, exist_ok=True)
    if SOURCE_PARQUET:
        supplied = Path(SOURCE_PARQUET)
        if not supplied.is_file():
            raise FileNotFoundError(supplied)
        if supplied.resolve() != SOURCE_PATH.resolve():
            shutil.copyfile(supplied, SOURCE_PATH)
    elif not SOURCE_PATH.exists():
        temporary = SOURCE_PATH.with_suffix('.download')
        try:
            with requests.get(SOURCE_URL, stream=True, timeout=(15, 90)) as response:
                response.raise_for_status()
                with temporary.open('wb') as stream:
                    for block in response.iter_content(1024 * 1024):
                        if block:
                            stream.write(block)
            if sha256_file(temporary) != SOURCE_SHA256:
                raise ValueError('Hash file tải về không khớp nguồn đã chốt.')
            temporary.replace(SOURCE_PATH)
        finally:
            temporary.unlink(missing_ok=True)
    if sha256_file(SOURCE_PATH) != SOURCE_SHA256:
        raise ValueError('Parquet không khớp nguồn đã chốt. Kiểm tra file Input.')
    print(f'Nguồn hợp lệ: {SOURCE_PATH} ({SOURCE_PATH.stat().st_size:,} byte)')


Nguồn hợp lệ: \kaggle\working\data\links_corpus.parquet (35,111,121 byte)


## 5. Chọn ID và tạo manifest

Nếu khoảng chọn có ID không tồn tại trong nguồn, notebook báo số lượng thiếu,
rồi lấy đúng các hàng thực sự có trong khoảng; không tạo ID hoặc URL thay thế.
Bước chuẩn bị hiện quét toàn file để kiểm tra nguồn và thống kê hostname, nên có thể mất một lúc.


In [5]:
with diagnostic_step('prepare'):
    import pyarrow.parquet as pq
    from collections import Counter
    from urllib.parse import urlsplit
    from IPython.display import display
    from datetime import datetime, timezone
    import time
    import uuid

    selected = pq.read_table(SOURCE_PATH, filters=[('id', '>=', ID_FROM), ('id', '<=', ID_TO)]).to_pylist()
    selected.sort(key=lambda row: row['id'])
    if not selected:
        raise ValueError('Không có URL nào trong khoảng ID đã chọn.')
    if len({row['id'] for row in selected}) != len(selected):
        raise ValueError('Nguồn có ID trùng trong khoảng chọn.')
    missing_count = ID_TO - ID_FROM + 1 - len(selected)
    print(f'Chọn {len(selected):,} URL; có {missing_count:,} ID không tồn tại trong khoảng.')
    print('Phân bố hostname:', dict(Counter(urlsplit(row['url']).hostname for row in selected)))
    display(selected[:5])

    LOG_DIR = RUN_DIR / 'logs'
    LOG_DIR.mkdir(exist_ok=True)

    def log_event(name, event):
        with (LOG_DIR / name).open('a', encoding='utf-8') as stream:
            stream.write(json.dumps(event, ensure_ascii=False) + '\n')

    def run_script(script, *arguments):
        # Output hiển thị trực tiếp để theo dõi tiến độ; không dùng shell.
        invocation_id = uuid.uuid4().hex
        started = time.monotonic()
        operation = str(arguments[0]) if script in (CRAWLER, CORPUS_RUNNER) else Path(script).stem
        log_event('invocations.jsonl', {
            'event': 'start', 'invocation_id': invocation_id, 'operation': operation,
            'at': datetime.now(timezone.utc).isoformat(), 'arguments': list(map(str, arguments)),
        })
        process = subprocess.Popen(
            [sys.executable, '-u', str(script), *map(str, arguments)],
            cwd=str(WORKDIR), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace', bufsize=1,
        )
        returncode = None
        try:
            with (LOG_DIR / (Path(script).stem + '.log')).open('a', encoding='utf-8') as log:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                    if operation == 'crawl':
                        try:
                            progress = json.loads(line)
                        except ValueError:
                            progress = {}
                        if isinstance(progress, dict) and 'completed_this_invocation' in progress:
                            log_event('progress.jsonl', {
                                **progress, 'invocation_id': invocation_id,
                                'at': datetime.now(timezone.utc).isoformat(),
                                'elapsed_seconds': time.monotonic() - started,
                            })
                            refresh_diagnostics()
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
        finally:
            process.stdout.close()
            log_event('invocations.jsonl', {
                'event': 'end', 'invocation_id': invocation_id, 'operation': operation,
                'at': datetime.now(timezone.utc).isoformat(),
                'elapsed_seconds': time.monotonic() - started,
                'returncode': process.returncode,
            })
            refresh_diagnostics()
        if returncode:
            raise subprocess.CalledProcessError(returncode, process.args)

    manifest_path = RUN_DIR / 'manifest.json'
    if not manifest_path.exists():
        run_script(CRAWLER, 'prepare', '--source', SOURCE_PATH, '--out', RUN_DIR,
                   '--size', len(selected), '--id-start', ID_FROM, '--id-stop', ID_TO + 1)
    else:
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        if manifest['source_sha256'] != SOURCE_SHA256 or manifest['selected'] != len(selected):
            raise ValueError('Manifest không khớp nguồn hoặc số URL đã chọn.')
        if Path(manifest['source']).resolve() != SOURCE_PATH.resolve():
            raise ValueError('Đường dẫn nguồn của run cũ khác notebook này; dùng run mới.')
        sample_bytes = (RUN_DIR / 'sample.jsonl').read_bytes()
        if hashlib.sha256(sample_bytes).hexdigest() != manifest['sample_sha256']:
            raise ValueError('Hash sample của run cũ đã thay đổi.')
        with (RUN_DIR / 'sample.jsonl').open(encoding='utf-8') as stream:
            sample = [json.loads(line) for line in stream if line.strip()]
        if {r['id']: r['url'] for r in sample} != {r['id']: r['url'] for r in selected} or len(sample) != len(selected):
            raise ValueError('ID/URL của run cũ không khớp khoảng chọn.')
        print('Manifest hợp lệ; giữ checkpoint để resume.')


Chọn 1,000 URL; có 0 ID không tồn tại trong khoảng.
Phân bố hostname: {'special.vietnamplus.vn': 1, 'bachmai.gov.vn': 999}


[{'id': 583, 'url': 'http://special.vietnamplus.vn/benh_vien_nhan_ai'},
 {'id': 584,
  'url': 'https://bachmai.gov.vn/bai-viet/-nguoi-dan-se-duoc-thu-huong-chat-luong-kham-chua-benh-quoc-te-ngay-tai-benh-vien-bach-mai--9867?id=da6d8b2d-c5bd-c612-e89c-e85f802063b4'},
 {'id': 585,
  'url': 'https://bachmai.gov.vn/bai-viet/-roi-loan-su-thich-ung-va-nhung-he-luy--9826?id=bda58e58-62e5-a99c-fd9f-aeb3142aa489'},
 {'id': 586,
  'url': 'https://bachmai.gov.vn/bai-viet/1-4-nguoi-truong-thanh-bi-tang-huyet-ap?id=6708761f-7a0c-9837-53c4-f202187b38c9'},
 {'id': 587,
  'url': 'https://bachmai.gov.vn/bai-viet/1-5-phai-manh-phan-nan-ve-tinh-trang-tren-bao-duoi-khong-nghe?id=eefea0b8-eef5-be7d-6426-88f0d9e5e491'}]

{"selected": 1000, "hosts": 97, "out": "\\kaggle\\working\\runs\\ids_583_1582_synced"}


## 6. Crawl — chạy lại ô này để tiếp tục

Tiến độ in mỗi 25 URL hoàn tất hoặc khi hết lượt. Chế độ robots theo `RESPECT_ROBOTS`,
giới hạn một request cùng registered domain, có retry và checkpoint theo ID.
`WORKERS = 4` không có nghĩa bốn request đồng thời đến cùng website.
ID đã có checkpoint, kể cả thất bại, sẽ được bỏ qua khi resume.
`MAX_NEW` giới hạn URL Requests mới; Selenium tiếp tục các URL lỗi đã có checkpoint
nhưng chưa thử browser. Lỗi cài Chrome được ghi riêng, vẫn giữ kết quả Requests.
`records/` là kết quả cuối dùng để audit/export; `requests_records/` và
`browser_records/` giữ bằng chứng của từng engine. Chỉ nội dung qua QC mới được xuất corpus.
ZIP `<run>_diagnostics.zip` được cập nhật sau mỗi log tiến độ và khi lệnh kết thúc,
kể cả ngắt ô crawl. File có một số mẫu record/raw để debug, không nén lại toàn bộ raw.


In [6]:
with diagnostic_step('crawl'):
    corpus_settings = {
        'user_agent': USER_AGENT, 'delay': DELAY, 'attempts': ATTEMPTS,
        'timeout': [PAGE_CONNECT_TIMEOUT, PAGE_READ_TIMEOUT], 'budget': PAGE_REQUEST_BUDGET,
        'respect_robots': RESPECT_ROBOTS,
        'robots_options': {'robots_attempts': ROBOTS_ATTEMPTS,
                           'robots_connect_timeout': ROBOTS_CONNECT_TIMEOUT,
                           'robots_read_timeout': ROBOTS_READ_TIMEOUT,
                           'robots_retry_budget': ROBOTS_RETRY_BUDGET,
                           'robots_failure_ttl': ROBOTS_FAILURE_TTL},
        'browser_mode': BROWSER_TEST_MODE, 'browser_user_agent': BROWSER_USER_AGENT,
        'chrome_binary': str(CHROME_BINARY) if CHROME_BINARY else None,
        'browser_load_timeout': BROWSER_LOAD_TIMEOUT, 'browser_wait_seconds': BROWSER_WAIT_SECONDS,
        'browser_attempts': BROWSER_ATTEMPTS,
    }
    settings_path = RUN_DIR / 'corpus_settings.json'
    settings_path.write_text(json.dumps(corpus_settings, ensure_ascii=False, indent=2), encoding='utf-8')
    arguments = ['crawl', '--out', RUN_DIR, '--workers', WORKERS, '--config', settings_path]
    if RECOVERY_RUN_PATH:
        arguments += ['--recovery-source', RECOVERY_RUN_PATH]
    if MAX_NEW is not None:
        arguments += ['--max-new', MAX_NEW]
    print('ZIP chẩn đoán tự cập nhật:', diagnostics_archive)
    run_script(CORPUS_RUNNER, *arguments)


ZIP chẩn đoán tự cập nhật: \kaggle\working\ids_583_1582_synced_diagnostics.zip
{"domain": "bachmai.gov.vn", "id": 584, "status": "extracted", "http": 200, "chars": 5141}
{"domain": "bachmai.gov.vn", "id": 585, "status": "extracted", "http": 200, "chars": 5023}
{"domain": "bachmai.gov.vn", "id": 586, "status": "extracted", "http": 200, "chars": 6817}
{"domain": "bachmai.gov.vn", "id": 587, "status": "extracted", "http": 200, "chars": 5072}
{"domain": "bachmai.gov.vn", "id": 588, "status": "extracted", "http": 200, "chars": 4200}
{"domain": "special.vietnamplus.vn", "id": 583, "status": "http_error", "http": 429, "chars": null}
{"completed": 1, "scheduled": 1000}
{"domain": "bachmai.gov.vn", "id": 589, "status": "extracted", "http": 200, "chars": 1598}
{"domain": "bachmai.gov.vn", "id": 590, "status": "extracted", "http": 200, "chars": 7018}
{"domain": "bachmai.gov.vn", "id": 591, "status": "extracted", "http": 200, "chars": 4442}
{"domain": "bachmai.gov.vn", "id": 592, "status": "extrac

KeyboardInterrupt: 

## 7. Báo cáo, kiểm tra và xuất corpus

Corpus accepted chỉ gồm tài liệu qua heuristic; chưa chứng minh chất lượng bằng chấm thủ công.
Audit chỉ chạy khi đủ checkpoint. Nếu đang crawl thử, vẫn xuất phần accepted hiện có;
manifest notebook ghi rõ đây là corpus chưa hoàn tất.


In [ ]:
with diagnostic_step('report_audit_export'):
    run_script(CRAWLER, 'report', '--out', RUN_DIR)
    summary = json.loads((RUN_DIR / 'quality_report.json').read_text(encoding='utf-8'))
    display({k: summary[k] for k in ['selected', 'completed', 'pending', 'accepted', 'review', 'rejected', 'status_counts']})
    if (RUN_DIR / 'access_summary.json').exists():
        display(json.loads((RUN_DIR / 'access_summary.json').read_text(encoding='utf-8')))
        print('Lỗi theo tổng URL dataset giảm dần:', RUN_DIR / 'WEB_ERRORS.md')
        print('Đối chiếu Requests/Selenium:', RUN_DIR / 'requests_vs_selenium.csv')
    if summary['pending'] == 0:
        run_script(AUDITOR, '--out', RUN_DIR)
    else:
        print('Run chưa hoàn tất; bỏ qua audit toàn mẫu. Chạy lại ô crawl để tiếp tục.')
    if summary['accepted'] > 0:
        run_script(EXPORTER, '--out', RUN_DIR)
        corpus = pq.read_table(RUN_DIR / 'corpus.parquet')
        print(corpus.schema)
        display(corpus.select(['id', 'url', 'title']).slice(0, 5).to_pylist())
    else:
        print('Chưa có tài liệu accepted; xem quarantine.jsonl và QUALITY_REPORT.md.')


## 8. ZIP chẩn đoán — tích hợp trong Output

Chạy ô này để tạo lại gói debug từ checkpoint hiện có, kể cả khi crawl chưa xong.
Không cần file Python riêng, không tải lại URL. ZIP chứa:
- `overview.html`, `summary.json`: tiến độ, tốc độ/ETA, trạng thái HTTP và QC.
- CSV từng ID, lỗi/pending, request/retry và redirect.
- Log pip/crawl, traceback notebook, manifest/cấu hình và phiên bản môi trường.
- Snapshot code, bằng chứng robots và tối đa 10 mẫu record/raw để kiểm tra parser.

Raw mẫu có ngân sách 10 MiB; các mẫu bị bỏ qua được ghi trong `sample_manifest.json`.
Giải nén ZIP và mở `overview.html`. Có thể gửi nguyên ZIP này để chẩn đoán lỗi.


In [ ]:
from IPython.display import FileLink
import zipfile

refresh_diagnostics(strict=True)
with zipfile.ZipFile(diagnostics_archive) as diagnostic_bundle:
    diagnostic_summary = json.loads(diagnostic_bundle.read('summary.json'))
display({key: diagnostic_summary[key] for key in ['selected', 'completed', 'pending', 'quality_status_counts', 'timing']})
print('ZIP chẩn đoán trong Output:', diagnostics_archive)
display(FileLink(str(diagnostics_archive.relative_to(WORKDIR))))


## 9. Đóng gói corpus đầy đủ để tải về

ZIP chứa cả raw, records, snapshot code và corpus để truy vết/resume; có thể khá lớn.
ZIP riêng `<run>_diagnostics.zip` nhỏ hơn: giải nén và mở `overview.html` để xem
tiến độ, tốc độ/ETA, HTTP status, lỗi và cờ chất lượng. Có CSV từng ID và log cài dependencies.
Tải file trong Output hoặc qua link bên dưới. Khi đổi session, cần giữ output hoặc
thêm run đã giải nén làm Kaggle Input và đặt `RESUME_RUN_PATH` trước khi chạy lại.
Resume không tự thử lại ID lỗi đã có checkpoint. Dùng `RECOVERY_RUN_PATH`
cùng `RUN_TAG` mới để thử lại lỗi bằng code/cấu hình mới và tái sử dụng raw thành công.


In [ ]:
from IPython.display import FileLink

notebook_manifest = {
    'id_from_inclusive': ID_FROM, 'id_to_inclusive': ID_TO,
    'dataset_revision': DATASET_REVISION, 'source_sha256': SOURCE_SHA256,
    'selected': summary['selected'], 'completed': summary['completed'],
    'pending': summary['pending'], 'accepted': summary['accepted'],
    'complete': summary['pending'] == 0,
    'packages': versions,
    'exported_files': [name for name in ['corpus.parquet', 'corpus.jsonl'] if (RUN_DIR / name).exists()],
}
(RUN_DIR / 'notebook_manifest.json').write_text(json.dumps(notebook_manifest, ensure_ascii=False, indent=2), encoding='utf-8')
refresh_diagnostics(strict=True)
archive = shutil.make_archive(str(WORKDIR / RUN_DIR.name), 'zip', root_dir=RUN_DIR.parent, base_dir=RUN_DIR.name)
print('ZIP:', archive)
print('Corpus:', RUN_DIR / 'corpus.parquet')
display(FileLink(str(Path(archive).relative_to(WORKDIR))))
print('ZIP chẩn đoán:', diagnostics_archive)
display(FileLink(str(diagnostics_archive.relative_to(WORKDIR))))
if (RUN_DIR / 'corpus.parquet').exists():
    display(FileLink(str((RUN_DIR / 'corpus.parquet').relative_to(WORKDIR))))
    display(FileLink(str((RUN_DIR / 'corpus.jsonl').relative_to(WORKDIR))))
